# CAMRA — Notebook 2: Pre-processing, training, and evaluation

Loads the aligned dataset built by `01_dataset_creation.ipynb` and runs the full experimental pipeline over ten seeds: pre-processing, CAMRA training (centralized model with the RL class-weighting controller, followed by federated adaptation), weighting-scheme and modality ablations, baselines, holdout evaluation, paired statistics, complexity and communication cost, and all diagnostic plots.

Set `ALIGNED_DATASET_PATH` and `OUT_DIR` in the configuration cell. To run a holdout configuration, set `EXCLUDE_ARCH` and/or `EXCLUDE_MALWARE_FAMILY`.

In [ ]:
!pip install scienceplots
!pip install thop

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# ═══════════════════════════════════════════════════════════
import os, gc, time, json, random, warnings
from math import pi
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from copy import deepcopy
from typing import Dict, List, Tuple, Optional, Set

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

from scipy.stats import skew as scipy_skew

from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from tqdm.auto import tqdm
from sklearn.metrics import (
    classification_report, confusion_matrix, recall_score, precision_score,
    f1_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    accuracy_score, silhouette_score, davies_bouldin_score,
    calinski_harabasz_score,
)

try:
    import scienceplots
    plt.style.use(['science', 'ieee', 'no-latex'])
    plt.rcParams.update({
        "text.usetex": False,
        "mathtext.fontset": "stix",
        "font.family": "serif"
    })
    # Addition on top of the settings above: 'ieee' alone prefers Times for
    # its serif font, which isn't installed on Kaggle/CI images (confirmed:
    # produced repeated "findfont... Times not found" warnings there even
    # with text.usetex off). DejaVu Serif ships WITH matplotlib itself, so
    # this always resolves cleanly instead of falling back with a warning.
    plt.rcParams['font.serif'] = ['DejaVu Serif']
except ImportError:
    plt.rcParams.update({
        "text.usetex": False,
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 300,
        "savefig.bbox": "tight"
    })
print(f"[INFO] Plot style: {'scienceplots (ieee, no-latex)' if 'scienceplots' in globals() else 'fallback rcParams (scienceplots unavailable)'}")

# Global rcParams for clean, simple, print-legible figures.

SCATTER_POINT_SIZE = 4          # was 5-10 in various scatter/UMAP calls; centralized here so it's one knob
SCATTER_ALPHA = 0.6

try:
    import umap as umap_lib
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("[WARN] umap-learn not installed - UMAP plots will be skipped")

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("[WARN] XGBoost not installed - XGB baseline will be skipped")

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
    print("[WARN] thop not installed - FLOPs will be estimated from parameter count instead of measured")

warnings.filterwarnings('ignore')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

def set_all_seeds(seed: int):
    """Reseeds every RNG that affects model training (weight init, dropout,
    batch shuffling, augmentation noise). The train/val/test SPLIT itself is
    computed ONCE using the fixed SEED above and is NOT re-randomized per
    seed - multi-seed runs are meant to isolate TRAINING variance (init +
    stochastic optimization), not conflate it with split variance. That's
    the standard, defensible reading of "multiple independent runs" in the
    reproducibility literature."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

## Configuration

Seeds, training budget, model and controller hyperparameters, federated-learning settings, the configurations to run, holdout settings, and plotting style.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — CONFIGURATION
# ═══════════════════════════════════════════════════════════

OUT_DIR = Path("/kaggle/working/results")   # must match Notebook 1's OUT_DIR
ALIGNED_DATASET_PATH = "/kaggle/input/datasets/advaitlr/cic26-post"
(OUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "reports").mkdir(parents=True, exist_ok=True)

ARCHS       = ["ARM", "MIPS", "MIPSEL", "x86"]
MODALITIES  = ["PCAP", "SAR", "STRACE"]

# --- sample-volume balance across modalities --------------------------------
# After hash-based alignment (Notebook 1) every execution contributes exactly
# one row, so we no longer discard whole executions to match the smallest
# modality. What we DO balance here is the FEATURE-COLUMN budget: STRACE's
# raw column count is usually far larger than PCAP's, and left unchecked its
# aggregated features alone would dominate the fused vector. This targets
# ~25% PCAP / ~25% SAR / ~50% STRACE of the total feature budget (selected
# via train-only ANOVA F-score).
MODALITY_FEATURE_SHARE = {"PCAP": 0.25, "SAR": 0.25, "STRACE": 0.50}
TOTAL_FEATURE_BUDGET    = 150       # total selected feature columns across all 3 modalities
SKEW_THRESHOLD          = 1.0       # |skew| above this (on TRAIN, non-negative cols) -> log1p

# --- class weighting ---------------------------------------------------------
# Fixed, train-set-derived class weights for the cross-entropy loss, applied
# identically everywhere a model is trained (central, ablation, FL, cross-
# arch, data-efficiency). Replaces the previous RL-driven adaptive weighting
# (see Cell 12/STEP 2 for why): weight_c = (N / (K * count_c)) ** power, then
# renormalized so the count-weighted average weight is 1 (keeps loss values
# on a comparable scale to unweighted training).
#   power=1.0 -> textbook "balanced" (full inverse frequency)
#   power=0.5 -> sqrt-dampened (default)
#   power=0.0 -> uniform (no reweighting, equivalent to the old torch.ones)
# On this dataset, power=1.0 gives a ~1554x ratio between the rarest and most
# common class (confirmed on real data) - enough for a single rare-class
# sample to dominate a batch's gradient direction and destabilize training,
# not just help the rare class. power=0.5 keeps the same ranking/distinction
# between rare classes while tempering the ratio to ~39x. Set to 1.0 if you
# want the literal, uncapped inverse-frequency weighting instead.
CLASS_WEIGHT_POWER = 0.5

# --- model / training --------------------------------------------------------
BATCH_SIZE   = 64
HIDDEN_DIM   = 128
EPOCHS_DL    = 50     # central DRL model AND ablation models both use this (epoch-matched)
FL_ROUNDS    = EPOCHS_DL
FL_LOCAL_EPOCHS = 2
ACCUM_STEPS  = max(1, 64 // BATCH_SIZE)
FL_DP_NOISE_SCALE = 1e-4

# Noise DECAY, added after confirming FedBN alone did not stop the loss
# climb on real data: FL_DP_NOISE_SCALE was being injected into every
# learnable parameter, every one of 50 rounds, with NO decay - an unchecked
# random walk in weight space that compounds round over round regardless of
# how well-converged the model already is. Exponentially decays from
# FL_DP_NOISE_SCALE (round 1) down to FL_DP_NOISE_SCALE *
# FL_DP_NOISE_FINAL_FRAC (the final round): early rounds get the same
# perturbation as before, but as training converges, less noise is injected,
# instead of continuing to perturb an already-good solution for 50 straight
# rounds.
FL_DP_NOISE_FINAL_FRAC = 0.05   # noise scale at the LAST round, as a fraction of FL_DP_NOISE_SCALE

def fl_noise_scale_for_round(round_: int, total_rounds: int) -> float:
    """1-indexed round number -> that round's DP noise scale, decaying
    exponentially from FL_DP_NOISE_SCALE to FL_DP_NOISE_SCALE *
    FL_DP_NOISE_FINAL_FRAC across `total_rounds`."""
    if total_rounds <= 1:
        return FL_DP_NOISE_SCALE
    decay_rate = FL_DP_NOISE_FINAL_FRAC ** (1.0 / (total_rounds - 1))
    return FL_DP_NOISE_SCALE * (decay_rate ** (round_ - 1))

# Round-level LR decay, added after confirming the loss climb persists (and
# is actually STEEPEST late in training, rounds ~43-50) even with FedBN and
# noise decay both active - the opposite of what either of those would
# predict if they were the dominant cause. `client_local_train` builds a
# FRESH CosineAnnealingLR every round that resets to full LR each time and
# anneals down only across that round's FL_LOCAL_EPOCHS - so the EFFECTIVE
# step size a client takes never shrinks round-over-round, no matter how
# converged the global model already is. Every client keeps taking
# full-sized local steps on an increasingly fine-tuned model for all 50
# rounds - a known contributor to FedAvg instability under non-IID clients,
# and consistent with a climb that WORSENS late in training. This decays the
# BASE local LR round-over-round, on top of (not instead of) the existing
# within-round cosine anneal.
FL_LOCAL_LR = 3e-4 * 0.5          # base local LR at round 1 (matches the previous fixed value)
FL_LOCAL_LR_FINAL_FRAC = 0.1      # local LR at the LAST round, as a fraction of FL_LOCAL_LR

def fl_lr_for_round(round_: int, total_rounds: int) -> float:
    """1-indexed round number -> that round's base local learning rate,
    decaying exponentially from FL_LOCAL_LR to FL_LOCAL_LR *
    FL_LOCAL_LR_FINAL_FRAC across `total_rounds`. client_local_train's own
    CosineAnnealingLR still anneals WITHIN each round, starting from
    whatever this function returns for that round."""
    if total_rounds <= 1:
        return FL_LOCAL_LR
    decay_rate = FL_LOCAL_LR_FINAL_FRAC ** (1.0 / (total_rounds - 1))
    return FL_LOCAL_LR * (decay_rate ** (round_ - 1))


# FedBN (Li et al., ICLR 2021): BatchNorm running_mean/running_var/weight/bias
# stay local to each client - never sent to the server, never averaged, never
# overwritten by the aggregated global model. Server-side FedAvg aggregation
# still happens for every other parameter (attention, encoders, classifier
# head) - this narrows WHAT gets aggregated, it does not remove aggregation
# or make this "not federated learning". Motivated by a real, confirmed
# pattern on this data: FL loss climbs steadily for 50 rounds while
# accuracy/F1 stay flat - the signature of growing miscalibration, and
# naively averaging 3 architectures' BN statistics together (statistics
# that describe none of them) is a documented cause of exactly this.
USE_FEDBN = True
N_CLASSES    = None   # populated once labels are known

USE_AUGMENTATION = True   # light Gaussian-noise augmentation on training batches
AUG_NOISE_STD    = 0.01

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
print(f"[INFO] Using device: {DEVICE}")

# --- multi-seed statistical evaluation ---------------------------------------
# Every trained model (central, FL, ablation, baselines, cross-arch,
# data-efficiency) is retrained once per seed below; final results report
# mean +/- std across seeds. The train/val/test SPLIT is fixed (see
# set_all_seeds above) - only training randomness varies. This is
# real compute: ~16 models/seed x len(SEEDS) for the deep configs alone,
# plus FL's (FL_ROUNDS x n_clients) local-training calls x len(SEEDS). Trim
# SEEDS for a quick smoke test; expand for the real reported run.
SEEDS = [42,16,29,34,56,67,71,86,99,46]
# SEEDS = [42,14]

# One-time diagnostic plots (UMAP, EDA-style) are illustrative, not a
# statistic to average - they run ONCE, on this representative seed's model,
# not once per seed.
REPRESENTATIVE_SEED = SEEDS[0]

# --- additional baselines -----------------------------------------------------
RUN_CNN_BASELINE = True    # 1D-CNN over the concatenated flattened feature vector
RUN_MLP_BASELINE = True    # plain feedforward network, same input

# --- class-weighting-scheme comparison ------------------------------------
# The MAIN central/FL models use RL-adaptive weighting (Cell 12). These two
# extra configs isolate whether the RL policy is earning its complexity, by
# training the SAME architecture/epoch budget with simpler alternatives:
RUN_FIXED_WEIGHT_COMPARISON = True   # static bundle["class_weights"], no RL adaptation
RUN_FOCAL_LOSS_COMPARISON   = True   # focal loss (gamma below), no explicit class weights
FOCAL_LOSS_GAMMA = 2.0

# --- modality holdout (test-time missing-modality robustness) ---------------
# Mirrors EXCLUDE_ARCH/EXCLUDE_MALWARE_FAMILY in spirit, but at TEST TIME on
# an already-trained model, not by retraining: zeroes out one modality's
# input (its own scaler's zero = "at the mean", the natural neutral value)
# and reports the resulting degradation. This simulates a sensor genuinely
# going missing at inference time, not a model trained without it.
RUN_MODALITY_HOLDOUT_EVAL = True

# --- optional leave-one-out generalization study ----------------------------
# Set either/both to test how well the model generalizes to an architecture
# or malware family it never trains on ("zero-day"-style evaluation). Leave
# both as None for the normal pipeline (no exclusion).


# "holdout": excluded rows are folded ENTIRELY into the real TEST set (never
#            train/val) - a leave-one-out generalization test embedded in
#            the actual evaluation (leave-one-architecture-out,
#            leave-one-malware-family-out). A separate
#            breakdown (held-out rows vs. the rest of test) is still
#            reported so that diagnostic value isn't lost.
# "drop":    excluded rows are removed from the dataset completely (never
#            used anywhere) - use this if you just want a smaller/cleaner run.
EXCLUSION_MODE = "holdout"   # "holdout" or "drop"

# Cells of the (Arch, MalwareFamily) grid with fewer than this many samples
# (checked on the REMAINING pool, after removing excluded rows) fall back to
# stratifying by MalwareFamily alone instead of jointly with Arch - some
# cells are too sparse (or exactly zero) to stratify jointly at all.
# Confirmed on real data: with MIPSEL+DarkNexus excluded, only Benign/Mirai
# clear this threshold in every remaining architecture; everything else
# (Gafgyt/Generic/Other) falls back automatically.
SPARSE_CELL_JOINT_THRESHOLD = 30

# Blend weight for the RL reward's "level" term (see Cell 12): 1.0 = pure
# absolute performance level (never vanishes, but doesn't reward improvement
# specifically), 0.0 = pure delta-vs-baseline (the old design - vanishes
# near convergence, so the reward collapses towards zero once training
# plateaus). 0.5 keeps both signals present throughout training.
REWARD_LEVEL_WEIGHT = 0.5



# ═══════════════════════════════════════════════════════════
# P1 — artifact paths, figure sizing, statistics settings
# ═══════════════════════════════════════════════════════════
ARTIFACT_DIR = OUT_DIR / "artifacts"
MANIFEST_DIR = OUT_DIR / "manifests"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

SAVE_EVAL_ARTIFACTS = True

# Paired statistics. Seeds are paired because the split is fixed and
# only training randomness varies, so every config sees the identical test set.
N_BOOTSTRAP   = 10000
BOOTSTRAP_CI  = 0.95

# Okabe-Ito colourblind-safe palette, ordered so the first four (the classes
# that actually carry samples) are maximally separable in both hue and
# greyscale value. Cycled together with linestyles so per-class curves stay
# distinguishable in a black-and-white print.
CLASS_PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7',
                 '#E69F00', '#56B4E9', '#000000', '#7F7F7F']
CLASS_LINESTYLES = ['-', '--', '-.', ':']

# Figures are generated at the width they are printed at, so the text is
# never shrunk when the figure is included in the paper.
PANEL_FIGSIZE = (3.3, 2.7)    # two-up subfigure (0.48\linewidth)
WIDE_FIGSIZE  = (6.8, 3.6)    # full-width figure
BASE_FONT_PT  = 8




# ═══════════════════════════════════════════════════════════
# A — stochastic policy settings
# ═══════════════════════════════════════════════════════════
RL_STOCHASTIC        = True    # False reproduces the old deterministic controller
RL_ALPHA_MIN         = 10.0     # concentration floor: alpha < 1 puts density
                               # spikes at the simplex corners, i.e. near
                               # single-class weighting, which is exactly the
                               # collapse the entropy term is meant to prevent
RL_ALPHA_MAX         = 60.0    # ceiling: as alpha_0 -> inf the sample converges
                               # on the mean and the policy degenerates back to
                               # the deterministic controller
RL_SAMPLE_FLOOR      = 1e-6    # Dirichlet samples can contain exact 0 in fp32,
                               # which makes log_prob -inf
RL_LOG_DIAGNOSTICS   = True

# ═══════════════════════════════════════════════════════════
# central-model loss and weighting prior
# ═══════════════════════════════════════════════════════════
CENTRAL_LOSS      = "focal"     # "ce"    : class-weighted cross-entropy (current behaviour)
                             # "focal" : focal loss with the RL weights as alpha
RL_RESIDUAL_PRIOR = True     # True : the policy outputs a multiplicative correction
                             #        on top of the frequency-derived class weights
                             # False: the policy outputs the weights from scratch
                             #        (current behaviour — starts at ~uniform)


# A class with fewer than this many val_reward samples cannot have its weight cut
# below the frequency prior (it can still be boosted). Reuses the split's own
# "too few samples to trust" threshold rather than introducing a new tuned knob.
RL_MIN_RELIABLE_SUPPORT = SPARSE_CELL_JOINT_THRESHOLD   # 30

RL_ENTROPY_BETA = 0.05   # weight on the Dirichlet policy entropy in the REINFORCE loss

# ═══════════════════════════════════════════════════════════
# Canonical result-row names — defined ONCE, used everywhere
# (ablation bar, JSON reports, statistics, final summary)
# ═══════════════════════════════════════════════════════════
ROW_DRL   = "Deep Reinforcement Learning"
ROW_FL    = "Federated Learning"
ROW_FIXED = "Fixed Weighting"
ROW_FOCAL = "Focal Loss"
ROW_RF    = "Random Forest"
ROW_XGB   = "XGBoost"
ROW_MLP   = "MLP"
ROW_CNN   = "CNN"
ROW_SINGLE = {"PCAP": "PCAP", "SAR": "SAR", "STRACE": "STRACE"}
ROW_PAIR   = {("PCAP", "SAR"):    "PCAP + SAR",
              ("PCAP", "STRACE"): "PCAP + STRACE",
              ("SAR", "STRACE"):  "SAR + STRACE"}

ROW_CATEGORY = {
    ROW_DRL: "proposed", ROW_FL: "proposed",
    ROW_FIXED: "weighting", ROW_FOCAL: "weighting",
    ROW_RF: "baseline", ROW_XGB: "baseline", ROW_MLP: "baseline", ROW_CNN: "baseline",
    **{name: "modality" for name in ROW_SINGLE.values()},
    **{name: "modality" for name in ROW_PAIR.values()},
}
EXPECTED_ROWS = list(ROW_CATEGORY)
assert len(EXPECTED_ROWS) == len(set(EXPECTED_ROWS)), "duplicate row name in ROW_* constants"

# The six pairwise comparisons among fixed weighting, focal loss, DRL, and FL.
STAT_CONTRASTS = [
    (ROW_DRL,   ROW_FIXED),
    (ROW_DRL,   ROW_FOCAL),
    (ROW_DRL,   ROW_FL),
    (ROW_FL,    ROW_FIXED),
    (ROW_FL,    ROW_FOCAL),
    (ROW_FIXED, ROW_FOCAL),
]

# A class with fewer test samples than this cannot give a meaningful F1 (the
# base-case test set has exactly one 'Other' sample). The primary macro-F1
# averages over classes with at least this many samples in the evaluated set;
# the all-class macro-F1 is kept alongside it. Any value from 2 to 9 excludes
# the same class here, so the result does not depend on the exact threshold.
MIN_TEST_SUPPORT = 5

# --- partial rerun (DRL + everything that consumes the DRL model) ------------
# True : rerun DRL, FL, modality holdout, holdout generalization and the
#        representative artifacts; splice every other row back in from the
#        previous run of the SAME configuration (same seeds, split, exclusions).
# False: full pipeline (use this for the combined arch x family runs).
RERUN_DRL_FL_ONLY = False

# ^ point these at the previous outputs of THIS run tag (base, exArch-ARM, ...)

SAVE_PER_SEED_STATE = False   # per-seed y_true/y_pred/y_prob + state_dicts for DRL and FL

In [ ]:
EXCLUDE_ARCH:            Optional[str] = None  # e.g. "x86" - must match a value in the Arch column
EXCLUDE_MALWARE_FAMILY:  Optional[str] = None  # e.g. "Tsunami" - must match a value in MalwareFamily
# Previous-run results; used only by the optional merge (RERUN_DRL_FL_ONLY) and the Focal Loss + Prior control.
PREV_DATA = Path("path/to/previous_run/reports")
PREV_RESULTS_JSON    = Path(f"{PREV_DATA}/multiseed_results.json")
PREV_COMPLEXITY_JSON = Path(f"{PREV_DATA}/computational_complexity.json")

## Load the aligned dataset

In [ ]:
if not Path(ALIGNED_DATASET_PATH).exists():
    raise FileNotFoundError(
        f"{ALIGNED_DATASET_PATH} not found. Run Notebook 1 (dataset_creation) first - "
        "it builds and saves this file."
    )
full_df = pd.read_parquet(ALIGNED_DATASET_PATH)
print(f"[INFO] Loaded aligned dataset -> {ALIGNED_DATASET_PATH}  (shape={full_df.shape})")
print(full_df["Arch"].value_counts())
print(full_df["MalwareFamily"].value_counts())

def get_modality_cols(df: pd.DataFrame) -> Dict[str, List[str]]:
    return {mod: [c for c in df.columns if c.startswith(f"{mod}__")] for mod in MODALITIES}

## Splitting and pre-processing utilities

Stratified train / val_select / val_reward / test split, and the pre-processing fitted on the training split only: log(1+x) transform of skewed columns, ANOVA-F feature selection under a per-modality budget, standard scaling, and frequency-based class weights.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — SPLIT + SKEW CORRECTION + FEATURE-BUDGET SELECTION + SCALING
# ═══════════════════════════════════════════════════════════

def compute_class_weights(y_train: np.ndarray, n_classes: int, power: float = CLASS_WEIGHT_POWER) -> torch.Tensor:
    """Fixed, train-set-derived cross-entropy weights: weight_c = (N / (K *
    count_c)) ** power, renormalized so the count-weighted average is 1. A
    class with ZERO training occurrences (only possible when
    EXCLUDE_MALWARE_FAMILY holds a class out entirely) gets a harmless
    placeholder weight of 1.0 - `weight[y]` is only ever indexed by labels
    that actually appear in a training batch, so a class that never occurs in
    training never has its weight value exercised at all."""
    counts = np.bincount(y_train, minlength=n_classes).astype(np.float64)
    N, K = counts.sum(), n_classes
    weights = np.ones(K, dtype=np.float64)
    nonzero = counts > 0
    weights[nonzero] = (N / (K * counts[nonzero])) ** power
    weighted_avg = (weights[nonzero] * counts[nonzero]).sum() / N
    weights[nonzero] = weights[nonzero] / weighted_avg   # count-weighted average -> 1
    return torch.tensor(weights, dtype=torch.float32)


def fit_log_transform(df: pd.DataFrame, cols: List[str], train_idx: np.ndarray,
                       threshold: float = SKEW_THRESHOLD) -> List[str]:
    """Decide which columns to log1p, using TRAIN rows only. A column is only
    ever log-transformed if its GLOBAL minimum is >= 0 (log1p is undefined
    below -1), so the same fitted transform is valid to apply to val/test.

    Only RIGHT-skewed columns (skew > threshold) are transformed. log1p
    compresses large values more than small ones, which reduces right-skew
    but makes left-skew (skew < -threshold) WORSE - confirmed on real data:
    a column with skew -2.19 became -8.07 after log1p. `abs(skew) >
    threshold` (the previous check) applied log1p in both directions and was
    silently making left-skewed columns worse."""
    log_cols = []
    for c in cols:
        train_vals = df[c].iloc[train_idx].to_numpy()
        if train_vals.std() < 1e-10:
            continue
        if df[c].min() >= 0 and scipy_skew(train_vals) > threshold:
            log_cols.append(c)
    return log_cols


def select_feature_budget(X_df: pd.DataFrame, y: np.ndarray, train_idx: np.ndarray,
                           modality_cols: Dict[str, List[str]],
                           share: Dict[str, float], total_budget: int) -> Dict[str, List[str]]:
    """Per-modality ANOVA-F feature selection, fit on TRAIN only, so no
    modality's raw column-count advantage (STRACE especially) dominates the
    fused feature vector. Target share is approximate: capped at the number
    of features actually available in that modality."""
    selected = {}
    for mod, cols in modality_cols.items():
        if not cols:
            selected[mod] = []
            continue
        k = max(1, min(len(cols), round(share.get(mod, 1.0 / len(modality_cols)) * total_budget)))
        skb = SelectKBest(f_classif, k=k)
        skb.fit(X_df[cols].iloc[train_idx].to_numpy(), y[train_idx])
        mask = skb.get_support()
        selected[mod] = [c for c, m in zip(cols, mask) if m]
    return selected


def _hybrid_strat_key(archs: np.ndarray, families: np.ndarray, threshold: int) -> np.ndarray:
    """Joint (Arch, MalwareFamily) stratification key where the cell has
    enough samples (>= threshold), falling back to MalwareFamily alone where
    it doesn't. Confirmed necessary on real data: some (Arch, Family) cells
    are as low as 2, or exactly 0 - too sparse (or empty) to stratify on
    jointly at all."""
    pair_counts = pd.Series(list(zip(archs, families))).value_counts()
    return np.array([
        f"{a}::{f}" if pair_counts[(a, f)] >= threshold else f"{f}"
        for a, f in zip(archs, families)
    ])


def _safe_stratified_split(idx: np.ndarray, strat_key: np.ndarray, test_size: float, seed: int):
    """train_test_split with a graceful fallback to unstratified when the
    smallest stratum is too small to split at all (sklearn requires every
    class to have at least 2 members; we require 3 to leave room for a
    further train/val split downstream)."""
    ok = pd.Series(strat_key).value_counts().min() >= 3
    return train_test_split(idx, test_size=test_size, random_state=seed,
                             stratify=strat_key if ok else None)


def build_model_ready_splits(full: pd.DataFrame):
    """Runs the whole leakage-free preprocessing chain and returns everything
    downstream steps need: per-modality train/val/test arrays, fitted
    scalers/selected-columns/log-columns (needed later to preprocess FL
    client data with the SAME fitted transform), the label encoder, and the
    execution-level split indices.

    If EXCLUDE_ARCH / EXCLUDE_MALWARE_FAMILY are set:
      - EXCLUSION_MODE="drop": matching rows are removed entirely (never
        train/val/test).
      - EXCLUSION_MODE="holdout": matching rows are folded ENTIRELY into the
        TEST set (never train/val) - this is a genuine leave-one-out
        generalization test embedded in the real evaluation, not a
        side-report. `bundle["idx_te_is_excluded"]` marks which test rows
        came from this exclusion vs. the regular stratified split, so both
        can still be reported separately.

    The remaining (non-excluded) pool is split via HYBRID stratification:
    jointly by (Arch, MalwareFamily) where a cell has enough samples
    (SPARSE_CELL_JOINT_THRESHOLD), falling back to MalwareFamily alone for
    sparser cells - see _hybrid_strat_key.
    """
    exclude_mask = pd.Series(False, index=full.index)
    if EXCLUDE_ARCH is not None:
        exclude_mask |= (full["Arch"] == EXCLUDE_ARCH)
    if EXCLUDE_MALWARE_FAMILY is not None:
        exclude_mask |= (full["MalwareFamily"] == EXCLUDE_MALWARE_FAMILY)

    if (EXCLUDE_ARCH is not None or EXCLUDE_MALWARE_FAMILY is not None) and not exclude_mask.any():
        print(f"[WARN] EXCLUDE_ARCH={EXCLUDE_ARCH!r} / EXCLUDE_MALWARE_FAMILY={EXCLUDE_MALWARE_FAMILY!r} "
              f"matched NO rows - check spelling against the actual values below:")
        print(f"    Arch values:            {sorted(full['Arch'].unique())}")
        print(f"    MalwareFamily values:   {sorted(full['MalwareFamily'].unique())}")

    if exclude_mask.any() and EXCLUSION_MODE == "drop":
        print(f"[INFO] EXCLUSION_MODE='drop': removing {int(exclude_mask.sum())} row(s) "
              f"(Arch={EXCLUDE_ARCH!r}, MalwareFamily={EXCLUDE_MALWARE_FAMILY!r}) entirely.")
        full = full[~exclude_mask].reset_index(drop=True)
        exclude_mask = pd.Series(False, index=full.index)   # nothing left to fold into test
    elif exclude_mask.any() and EXCLUSION_MODE not in ("holdout", "drop"):
        raise ValueError(f"Unknown EXCLUSION_MODE={EXCLUSION_MODE!r}, expected 'drop' or 'holdout'")
    elif exclude_mask.any():
        print(f"[INFO] EXCLUSION_MODE='holdout': {int(exclude_mask.sum())} row(s) "
              f"(Arch={EXCLUDE_ARCH!r}, MalwareFamily={EXCLUDE_MALWARE_FAMILY!r}) will be folded entirely "
              f"into the TEST set (never train/val) - a leave-one-out generalization test embedded in "
              f"the real evaluation.")

    le = LabelEncoder().fit(full["MalwareFamily"])
    y_all = le.transform(full["MalwareFamily"])
    class_names = le.classes_.tolist()
    n_classes = len(class_names)

    idx_all = np.arange(len(full))
    arch_all = full["Arch"].to_numpy()
    excl = exclude_mask.to_numpy()
    idx_excluded  = idx_all[excl]
    idx_remaining = idx_all[~excl]

    # ---- hybrid-stratified split of the REMAINING (non-excluded) pool ----
    strat_key_full = _hybrid_strat_key(arch_all[idx_remaining], full["MalwareFamily"].to_numpy()[idx_remaining],
                                        SPARSE_CELL_JOINT_THRESHOLD)
    n_joint = int(pd.Series(strat_key_full).str.contains("::").sum())
    print(f"[INFO] Hybrid stratification: {n_joint}/{len(strat_key_full)} remaining rows use joint "
          f"(Arch, MalwareFamily) keys (cell >= {SPARSE_CELL_JOINT_THRESHOLD}); "
          f"the rest fall back to MalwareFamily-only.")

    sub_all = np.arange(len(idx_remaining))
    sub_tr, sub_te = _safe_stratified_split(sub_all, strat_key_full, test_size=0.2, seed=SEED)
    sub_tr2, sub_val = _safe_stratified_split(sub_tr, strat_key_full[sub_tr], test_size=0.15, seed=SEED)

    idx_tr  = idx_remaining[sub_tr2]
    idx_val = idx_remaining[sub_val]
    idx_te  = np.concatenate([idx_remaining[sub_te], idx_excluded])
    idx_te_is_excluded = np.concatenate([np.zeros(len(sub_te), dtype=bool), np.ones(len(idx_excluded), dtype=bool)])

    # ---- split val further into val_select (hyperparameter selection and
    # monitoring) and val_reward (RL controller state and reward), so the data
    # that train the controller are never used for model decisions.
    # Stratified by MalwareFamily alone (val is already small; joint
    # Arch+Family stratification at half this size would be too sparse).
    val_families = y_all[idx_val]
    val_strat_ok = np.unique(val_families, return_counts=True)[1].min() >= 2 if len(idx_val) else False
    sub_val_sel, sub_val_rew = train_test_split(
        np.arange(len(idx_val)), test_size=0.5, random_state=SEED,
        stratify=val_families if val_strat_ok else None
    )
    idx_val_select = idx_val[sub_val_sel]
    idx_val_reward = idx_val[sub_val_rew]

    # leakage sanity check: hash-level disjointness across every split
    h = full["hash"].to_numpy()
    parts = {"train": idx_tr, "val_select": idx_val_select, "val_reward": idx_val_reward, "test": idx_te}
    names = list(parts.keys())
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            assert len(set(h[parts[a]]) & set(h[parts[b]])) == 0, f"Hash leakage between {a} and {b}!"

    modality_cols = get_modality_cols(full)
    X_full = full.copy()
    all_feat_cols = [c for cols in modality_cols.values() for c in cols]

    log_cols = fit_log_transform(X_full, all_feat_cols, idx_tr)
    for c in log_cols:
        X_full[c] = np.log1p(X_full[c].clip(lower=0))

    selected_cols = select_feature_budget(X_full, y_all, idx_tr, modality_cols,
                                           MODALITY_FEATURE_SHARE, TOTAL_FEATURE_BUDGET)

    scalers, X = {}, {"train": {}, "val_select": {}, "val_reward": {}, "test": {}}
    for mod, cols in selected_cols.items():
        if not cols:
            for split in X:
                X[split][mod] = np.zeros((0, 0), dtype=np.float32)
            continue
        sc = StandardScaler().fit(X_full[cols].iloc[idx_tr].to_numpy(dtype=np.float32))
        scalers[mod] = sc
        for split, ix in [("train", idx_tr), ("val_select", idx_val_select),
                           ("val_reward", idx_val_reward), ("test", idx_te)]:
            arr = sc.transform(X_full[cols].iloc[ix].to_numpy(dtype=np.float32))
            X[split][mod] = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

    y = {"train": y_all[idx_tr], "val_select": y_all[idx_val_select],
         "val_reward": y_all[idx_val_reward], "test": y_all[idx_te]}
    dims = {mod: X["train"][mod].shape[1] for mod in MODALITIES}

    print(f"[INFO] Executions -> train:{len(idx_tr)}  val_select:{len(idx_val_select)}  "
          f"val_reward:{len(idx_val_reward)}  test:{len(idx_te)} "
          f"({int(idx_te_is_excluded.sum())} of which are the held-out arch/malware rows)")
    print(f"[INFO] Feature dims (post budget-selection) -> "
          f"PCAP:{dims['PCAP']}  SAR:{dims['SAR']}  STRACE:{dims['STRACE']}")
    print(f"[INFO] log1p-transformed {len(log_cols)}/{len(all_feat_cols)} columns "
          f"(train-fit |skew|>{SKEW_THRESHOLD}, non-negative only)")

    cw = compute_class_weights(y["train"], n_classes)
    print(f"[INFO] Class weights (power={CLASS_WEIGHT_POWER}, train-set-derived, fixed for all training below):")
    for name, w, cnt in zip(class_names, cw.tolist(), np.bincount(y["train"], minlength=n_classes)):
        print(f"    {name:<12} n_train={int(cnt):<6} weight={w:.3f}")

    bundle = {
        "X": X, "y": y, "dims": dims, "le": le, "class_names": class_names, "n_classes": n_classes,
        "idx_tr": idx_tr, "idx_val_select": idx_val_select, "idx_val_reward": idx_val_reward, "idx_te": idx_te,
        "idx_te_is_excluded": idx_te_is_excluded,
        "scalers": scalers, "selected_cols": selected_cols, "log_cols": log_cols,
        "arch_of_row": full["Arch"].to_numpy(), "hash_of_row": h,
        "class_weights": cw,
    }
    return bundle



def transform_with_fitted(full: pd.DataFrame, bundle: Dict, row_idx: np.ndarray) -> Dict[str, np.ndarray]:
    """Applies the ALREADY-FITTED (train-only) log1p + scaler to an arbitrary
    subset of rows from `full` (used for FL clients / cross-arch evaluation,
    so no client ever re-fits its own scaler on data that includes test
    rows)."""
    X_full = full.copy()
    for c in bundle["log_cols"]:
        X_full[c] = np.log1p(X_full[c].clip(lower=0))
    out = {}
    for mod, cols in bundle["selected_cols"].items():
        if not cols:
            out[mod] = np.zeros((len(row_idx), 0), dtype=np.float32)
            continue
        arr = bundle["scalers"][mod].transform(X_full[cols].iloc[row_idx].to_numpy(dtype=np.float32))
        out[mod] = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return out

## Model architecture

Datasets and the CAMRA network: modality encoders, cross-modal attention, and classification head.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — DATASETS & MODEL ARCHITECTURE
# ═══════════════════════════════════════════════════════════

class MultimodalDataset(Dataset):
    def __init__(self, X_pcap, X_sar, X_strace, y, augment=False, noise_std=0.02):
        self.X_pcap   = torch.tensor(X_pcap,   dtype=torch.float32)
        self.X_sar    = torch.tensor(X_sar,    dtype=torch.float32)
        self.X_strace = torch.tensor(X_strace, dtype=torch.float32)
        self.y        = torch.tensor(y,        dtype=torch.long)
        self.augment  = augment
        self.noise_std = noise_std
        self.n = len(y)

    def __len__(self): return self.n

    def __getitem__(self, idx):
        xp, xs, xt = self.X_pcap[idx], self.X_sar[idx], self.X_strace[idx]
        if self.augment:
            xp = xp + torch.randn_like(xp) * self.noise_std
            xs = xs + torch.randn_like(xs) * self.noise_std
            xt = xt + torch.randn_like(xt) * self.noise_std
        return xp, xs, xt, self.y[idx]

class SingleModalDataset(Dataset):
    def __init__(self, X, y, augment=False, noise_std=0.02):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment, self.noise_std = augment, noise_std

    def __len__(self): return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.augment: x = x + torch.randn_like(x) * self.noise_std
        return x, self.y[idx]

class TabularResidualBlock(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.3):
        super().__init__()
        self.fc1, self.bn1 = nn.Linear(dim, dim), nn.BatchNorm1d(dim)
        self.fc2, self.bn2 = nn.Linear(dim, dim), nn.BatchNorm1d(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = self.drop(F.gelu(self.bn1(self.fc1(x))))
        return F.gelu(x + self.bn2(self.fc2(h)))

class ModalityEncoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 128, dropout: float = 0.3):
        super().__init__()
        self.proj   = nn.Linear(input_dim, hidden_dim)
        self.bn_in  = nn.BatchNorm1d(hidden_dim)
        self.res1   = TabularResidualBlock(hidden_dim, dropout)
        self.res2   = TabularResidualBlock(hidden_dim, dropout)

    def forward(self, x):
        return self.res2(self.res1(F.gelu(self.bn_in(self.proj(x)))))

class CrossModalAttention(nn.Module):
    def __init__(self, dim: int = 128, n_heads: int = 4):
        super().__init__()
        n_heads = max(1, min(n_heads, dim // 8)) if dim < 64 else n_heads
        drop = 0.1 if dim >= 32 else 0.0
        self.attn  = nn.MultiheadAttention(dim, n_heads, batch_first=True, dropout=drop)
        self.ff    = nn.Sequential(nn.Linear(dim, dim * 2), nn.GELU(), nn.Dropout(drop), nn.Linear(dim * 2, dim))
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, tokens):
        attn_out, _ = self.attn(tokens, tokens, tokens)
        tokens = self.norm1(tokens + attn_out)
        return self.norm2(tokens + self.ff(tokens))

class IoTMalwareDetector(nn.Module):
    def __init__(self, pcap_dim, sar_dim, strace_dim, hidden=128, n_classes=7, dropout=0.3):
        super().__init__()
        self.enc_pcap   = ModalityEncoder(pcap_dim,   hidden, dropout)
        self.enc_sar    = ModalityEncoder(sar_dim,    hidden, dropout)
        self.enc_strace = ModalityEncoder(strace_dim, hidden, dropout)
        self.fusion     = CrossModalAttention(hidden, n_heads=max(1, hidden // 32))
        mlp_in = hidden * 3
        self.classifier = nn.Sequential(
            nn.Linear(mlp_in, max(32, mlp_in // 2)), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(max(32, mlp_in // 2), max(16, mlp_in // 4)), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(max(16, mlp_in // 4), n_classes)
        )

    def forward(self, x_pcap, x_sar, x_strace):
        tokens = torch.stack([self.enc_pcap(x_pcap), self.enc_sar(x_sar), self.enc_strace(x_strace)], dim=1)
        return self.classifier(self.fusion(tokens).flatten(1))

    def get_embeddings(self, x_pcap, x_sar, x_strace):
        tokens = torch.stack([self.enc_pcap(x_pcap), self.enc_sar(x_sar), self.enc_strace(x_strace)], dim=1)
        return self.fusion(tokens).flatten(1)

class SingleModalModel(nn.Module):
    def __init__(self, input_dim, n_classes, hidden=128, dropout=0.3):
        super().__init__()
        self.enc = ModalityEncoder(input_dim, hidden, dropout)
        self.clf = nn.Sequential(
            nn.Linear(hidden, max(16, hidden // 2)), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(max(16, hidden // 2), n_classes)
        )

    def forward(self, x): return self.clf(self.enc(x))

class MLPBaseline(nn.Module):
    """Plain feedforward network on the concatenated flattened feature vector
    (all three modalities' selected features, hstacked) - a deep baseline
    with none of IoTMalwareDetector's structural priors (no per-modality
    encoders, no cross-modal attention), to compare against a simpler deep
    baseline, not just RF/XGBoost."""
    def __init__(self, input_dim: int, n_classes: int, hidden: int = 256, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, hidden // 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden // 4, n_classes),
        )
    def forward(self, x): return self.net(x)

class TabularCNN(nn.Module):
    """1D-CNN over the concatenated flattened feature vector, treating it as
    a length-`input_dim` sequence with 1 input channel. Convolutions capture
    LOCAL interactions between adjacent selected features; this is a standard
    way to give a CNN baseline a fair shot on tabular data (there's no
    natural 2D spatial structure to exploit here, unlike images)."""
    def __init__(self, input_dim: int, n_classes: int, dropout: float = 0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2), nn.BatchNorm1d(32), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 64, kernel_size=3, padding=1), nn.BatchNorm1d(64), nn.GELU(),
            nn.AdaptiveAvgPool1d(4),
        )
        self.clf = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 4, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )
    def forward(self, x):
        return self.clf(self.conv(x.unsqueeze(1)))   # (batch, features) -> (batch, 1, features)

## RL class-weighting controller

Dirichlet policy that samples per-class multipliers on the frequency prior at every epoch, trained with REINFORCE. The reward combines the level and the change of the smoothed per-class recall on val_reward.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12 — RL AGENT (stochastic policy, REINFORCE)
# ═══════════════════════════════════════════════════════════

class RLPolicyNet(nn.Module):
    """Maps the training state to the concentration parameters of a Dirichlet
    over the class-weight simplex.

    The previous version returned a softmax that was used directly as the
    action. A softmax output is a point on the simplex, not a distribution over
    the simplex, so there was no density to evaluate and therefore no valid
    log-likelihood. Emitting concentrations instead gives an explicit
    distribution the action is drawn from, which is what the policy-gradient
    estimator requires."""

    def __init__(self, state_dim: int, n_classes: int,
                 alpha_min: Optional[float] = None, alpha_max: Optional[float] = None):
        super().__init__()
        h = max(32, n_classes * 4)
        self.net = nn.Sequential(
            nn.Linear(state_dim, h), nn.Tanh(),
            nn.Linear(h, h // 2), nn.Tanh(),
            nn.Linear(h // 2, n_classes)
        )
        # resolved at construction, not at class definition
        self.alpha_min = RL_ALPHA_MIN if alpha_min is None else alpha_min
        self.alpha_max = RL_ALPHA_MAX if alpha_max is None else alpha_max
        
    def forward(self, s):
        # softplus keeps alpha strictly positive; the floor keeps the density
        # away from the simplex corners and the ceiling stops the policy from
        # silently annealing into a deterministic map.
        return self.alpha_min + F.softplus(self.net(s)).clamp(max=self.alpha_max)


class RLAgent:
    """Single-step contextual bandit over per-class training weights.

    Each epoch the agent observes a state (per-class recall on val_reward plus
    four scalar summaries), samples a weight vector from Dirichlet(alpha(s)),
    the classifier trains one epoch under those weights, and the resulting
    recall yields a scalar reward. The policy is updated with REINFORCE against
    an exponential-moving-average baseline.

    Deliberately not framed as sequential control: one epoch's weighting does
    not induce a state the policy is then obliged to act from, so there is no
    meaningful discount factor and no trajectory to credit. A bandit is the
    correct formalism and the honest one."""

    def __init__(self, n_classes: int, lr: float = 1e-3, entropy_beta: float = 0.05,
                 reward_ema_decay: float = 0.7, stochastic: Optional[bool] = None,
                 prior: Optional[torch.Tensor] = None,
                 support: Optional[np.ndarray] = None):
        self.n_cls, self.entropy_beta = n_classes, entropy_beta
        self.reward_ema_decay = reward_ema_decay
        self.stochastic = RL_STOCHASTIC if stochastic is None else bool(stochastic)
        self.prior = (torch.ones(n_classes) if prior is None
                      else prior.detach().float().clone()).to(DEVICE)

        # Per-class sample count of the split the reward is measured on.
        #   measurable : at least one sample -> recall means something
        #   reliable   : enough samples that a LOW weight can be trusted
        self.support = None if support is None else np.asarray(support, dtype=float)
        if self.support is None:
            measurable = np.ones(n_classes, dtype=bool)
            reliable   = np.ones(n_classes, dtype=bool)
        else:
            measurable = self.support > 0
            reliable   = self.support >= RL_MIN_RELIABLE_SUPPORT
        self.measurable = measurable
        self._t_measurable = torch.as_tensor(measurable, device=DEVICE)
        self._t_reliable   = torch.as_tensor(reliable,   device=DEVICE)

        self.policy = RLPolicyNet(n_classes + 4, n_classes).to(DEVICE)
        self.optim  = torch.optim.Adam(self.policy.parameters(), lr=lr)
        self.log_probs, self.entropies, self.rewards = [], [], []
        self.reward_history, self.recall_history = [], []
        self.prev_recall_signal, self._baseline = 0.0, 0.0
        self.alpha0_history, self.entropy_history, self.explore_gap_history, self.alpha_history = [], [], [], []

    def _smoothed_recall(self, class_recall: np.ndarray) -> np.ndarray:
        """Laplace-smoothed recall (TP + 1) / (n + 2): the posterior mean under a
        uniform prior. With 4 samples a perfect score reads 0.83 instead of 1.0;
        with 576 samples the change is ~0.002. A class with no samples reads a
        neutral 0.5 instead of recall_score's zero_division=0, which reported it
        as completely missed every epoch."""
        if self.support is None:
            return class_recall
        n = self.support
        return (class_recall * n + 1.0) / (n + 2.0)

    def _state(self, class_recall: np.ndarray, epoch_frac: float, val_loss: float):
        r = self._smoothed_recall(class_recall)
        p = (r + 1e-8) / (r.sum() + 1e-8)
        entropy = float(-np.sum(p * np.log(p + 1e-8)))
        return torch.tensor(
            np.concatenate([r, [self.prev_recall_signal, epoch_frac, val_loss, entropy]]),
            dtype=torch.float32, device=DEVICE
        )

    def get_weights(self, class_recall: np.ndarray, epoch_frac: float, val_loss: float,
                    stochastic: Optional[bool] = None) -> torch.Tensor:
        """Action w ~ Dirichlet(alpha(s)); class weights = prior * m, where the
        multiplier m = K*w is then constrained by the evidence available:
          - no val_reward samples     -> m = 1   (no evidence, no correction)
          - fewer than the threshold  -> m >= 1  (can boost, cannot cut)
        The constraint is applied to the sampled action by the environment, so
        log_prob remains that of the sampled Dirichlet action and the
        score-function estimator stays valid."""
        use_stoch = self.stochastic if stochastic is None else stochastic
        s = self._state(class_recall, epoch_frac, val_loss)
        alpha = self.policy(s)
        dist = torch.distributions.Dirichlet(alpha, validate_args=False)

        if use_stoch:
            w = dist.sample()
            w = w.clamp_min(RL_SAMPLE_FLOOR)
            w = w / w.sum()
            self.log_probs.append(dist.log_prob(w))
            self.entropies.append(dist.entropy())
        else:
            w = alpha / alpha.sum()

        if RL_LOG_DIAGNOSTICS:
            mean_w = (alpha / alpha.sum()).detach()
            self.alpha0_history.append(float(alpha.sum().detach().cpu()))
            self.entropy_history.append(float(dist.entropy().detach().cpu()))
            self.explore_gap_history.append(float((w - mean_w).abs().sum().cpu()))
            self.alpha_history.append(alpha.detach().cpu().numpy().astype(float).tolist())

        m = w * self.n_cls
        m = torch.where(self._t_reliable,   m, torch.clamp(m, min=1.0))
        m = torch.where(self._t_measurable, m, torch.ones_like(m))
        return (m * self.prior).detach()

    def store_reward(self, class_recall: np.ndarray, level_weight: float = REWARD_LEVEL_WEIGHT):
        """reward = mean over MEASURABLE classes of (2 * smoothed_recall - 1),
        blended level + delta as before. Classes with no val_reward samples are
        left out: including them added a constant -1 that no action could change."""
        r = self._smoothed_recall(class_recall)
        recall_signal = float(np.mean(2.0 * r[self.measurable] - 1.0))
        delta_term = recall_signal - self.prev_recall_signal
        reward = level_weight * recall_signal + (1 - level_weight) * delta_term
        self.rewards.append(reward)
        self.reward_history.append(reward)
        self.recall_history.append(recall_signal)
        self.prev_recall_signal = (self.reward_ema_decay * self.prev_recall_signal +
                                   (1 - self.reward_ema_decay) * recall_signal)
    def update(self):
        """REINFORCE with an EMA baseline and entropy regularisation:

            L = - sum_t log pi(a_t | s_t) * (R_t - b)  -  beta * sum_t H[pi(.|s_t)]

        `- log_prob * advantage` is now a genuine score-function estimator
        because `log_prob` is the Dirichlet log-density at the sampled action."""
        if not self.rewards or not self.log_probs:
            self.rewards.clear(); self.log_probs.clear(); self.entropies.clear()
            return
        R = torch.tensor(self.rewards, dtype=torch.float32, device=DEVICE)
        self._baseline = 0.9 * self._baseline + 0.1 * float(R.mean())
        adv = R - self._baseline
        if adv.numel() > 1 and float(adv.std()) > 1e-8:
            adv = adv / (adv.std() + 1e-8)

        logp = torch.stack(self.log_probs)
        ent  = torch.stack(self.entropies)
        loss = -(logp * adv).sum() - self.entropy_beta * ent.sum()

        self.optim.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
        self.optim.step()
        self.log_probs.clear(); self.entropies.clear(); self.rewards.clear()

## Federated learning utilities

FedAvg aggregation (BatchNorm statistics kept local) and the parameter-noise regularizer.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 13 — FEDERATED LEARNING UTILITIES
# ═══════════════════════════════════════════════════════════

def add_dp_noise(model: nn.Module, noise_scale: float):
    if noise_scale <= 0.0: return
    with torch.no_grad():
        for param in model.parameters():
            param.add_(torch.randn_like(param) * noise_scale)

def get_batchnorm_keys(model: nn.Module) -> Set[str]:
    """State-dict keys belonging to BatchNorm1d layers (weight, bias,
    running_mean, running_var, num_batches_tracked). Used to implement
    FedBN: these stay LOCAL to each client, never averaged across clients
    and never overwritten by the server - unlike Linear/attention weights,
    a BatchNorm running_mean/running_var IS that client's own local feature
    statistics (e.g. "MIPS's average syscall-count magnitude"); averaging
    three architectures' statistics together describes none of them
    accurately. Non-BN parameters (everything else) are still fully
    aggregated via FedAvg - this narrows WHAT gets aggregated, it doesn't
    remove server-side aggregation."""
    keys = set()
    for name, module in model.named_modules():
        if isinstance(module, nn.BatchNorm1d):
            for suffix in ("weight", "bias", "running_mean", "running_var", "num_batches_tracked"):
                keys.add(f"{name}.{suffix}")
    return keys


def fedavg_aggregate(global_model: nn.Module, client_models: List[nn.Module],
                     client_sizes: List[int], bn_keys: Optional[Set[str]] = None) -> nn.Module:
    bn_keys = bn_keys or set()
    total = sum(client_sizes)
    g_state = global_model.state_dict()
    for key in g_state:
        if key in bn_keys:
            continue   # FedBN: skip - BN stats are never averaged, stay local per client
        g_state[key] = sum(
            client_models[i].state_dict()[key].float() * (client_sizes[i] / total)
            for i in range(len(client_models))
        )
    global_model.load_state_dict(g_state)
    return global_model

def client_local_train(model: nn.Module, loader: DataLoader, class_weights: torch.Tensor,
                       epochs: int = FL_LOCAL_EPOCHS, lr: float = 1e-4, dp_noise: float = 0.0,
                       loss_mode: Optional[str] = None) -> nn.Module:
    """Local client update. loss_mode defaults to CENTRAL_LOSS so the FL phase
    trains with the same objective as the central model it starts from."""
    loss_mode = CENTRAL_LOSS if loss_mode is None else loss_mode
    model = model.to(DEVICE).train()
    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sc   = GradScaler(enabled=USE_AMP)
    cw   = class_weights.to(DEVICE)
    if loss_mode == "focal":
        crit = lambda logits, y: focal_loss(logits, y, gamma=FOCAL_LOSS_GAMMA, alpha=cw)
    else:
        crit = nn.CrossEntropyLoss(weight=cw)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    for _ in range(epochs):
        for batch in loader:
            opt.zero_grad()
            with autocast(enabled=USE_AMP):
                logits = model(*[b.to(DEVICE) for b in batch[:-1]])
                loss = crit(logits, batch[-1].to(DEVICE))
            sc.scale(loss).backward()
            sc.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            sc.step(opt); sc.update()
        sched.step()
    add_dp_noise(model, dp_noise)
    return model.cpu()

## Training and evaluation utilities

Training loop, evaluation, focal loss, metrics, complexity measurement (parameters, FLOPs, size, latency), and the test-time modality-holdout evaluator.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 14 — TRAINING / EVALUATION UTILITIES
# ═══════════════════════════════════════════════════════════

def get_model_size_mb(model: nn.Module) -> float:
    """Serialized state_dict size, matching what you'd actually ship/deploy -
    more accurate than parameter_count * 4 bytes since it includes buffers
    (BatchNorm running stats, etc.) exactly as torch.save would write them."""
    import io
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.tell() / (1024 ** 2)


def count_flops_and_params(model: nn.Module, sample_inputs: tuple) -> Tuple[float, int]:
    """FLOPs for one forward pass at batch size 1, plus total parameter
    count. Uses thop when available (measures the actual graph); falls back
    to a rough parameter-based estimate (2x params, the standard rule-of-
    thumb for a forward pass of mostly-linear layers) if thop isn't
    installed, clearly labeled as an estimate either way."""
    n_params = sum(p.numel() for p in model.parameters())
    was_training = model.training
    model.eval()
    if HAS_THOP:
        try:
            flops, _ = thop_profile(model, inputs=sample_inputs, verbose=False)
        except Exception as e:
            print(f"[WARN] thop FLOPs profiling failed ({e}); falling back to parameter-based estimate")
            flops = 2.0 * n_params
    else:
        flops = 2.0 * n_params
    if was_training:
        model.train()
    return float(flops), int(n_params)


@torch.no_grad()
def measure_inference_latency(model: nn.Module, sample_inputs: tuple, n_warmup: int = 10, n_runs: int = 100) -> float:
    """Median per-sample latency (ms) at batch size 1, the deployment-
    relevant number (a single IoT device classifying one execution at a
    time), not a large-batch throughput figure that hides real-world
    per-request cost. Median instead of mean - robust to the odd slow first
    call/GC pause rather than being dragged by it."""
    model.eval()
    for _ in range(n_warmup):
        model(*sample_inputs)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model(*sample_inputs)
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))


def report_model_complexity(model: nn.Module, sample_inputs: tuple, name: str) -> Dict:
    """One-stop complexity report: params, FLOPs, size, inference latency -
    training time is excluded, as it is
    timed separately around each training call (see time_call below)."""
    flops, n_params = count_flops_and_params(model, sample_inputs)
    size_mb = get_model_size_mb(model)
    latency_ms = measure_inference_latency(model, sample_inputs)
    report = {
        "model": name, "params": n_params, "flops": flops,
        "size_mb": size_mb, "latency_ms_per_sample": latency_ms,
    }
    print(f"[COMPLEXITY] {name:<20} params={n_params:>10,}  FLOPs={flops:>14,.0f}  "
          f"size={size_mb:>7.3f} MB  latency={latency_ms:>7.4f} ms/sample")
    return report


def time_call(fn, *args, **kwargs) -> Tuple[object, float]:
    """Wraps any training call, returning (result, elapsed_seconds). Used to
    report training time per model/config."""
    t0 = time.time()
    result = fn(*args, **kwargs)
    return result, time.time() - t0


def focal_loss(logits: torch.Tensor, targets: torch.Tensor, gamma: float = FOCAL_LOSS_GAMMA,
               alpha: Optional[torch.Tensor] = None) -> torch.Tensor:
    """Multi-class focal loss: down-weights well-classified (high-confidence-
    correct) examples so training focuses on hard/misclassified ones,
    without needing explicit inverse-frequency class weights. Used only for
    the weighting-scheme comparison configs.
    `alpha` (optional per-class weight) lets focal loss and class-weighted
    CE be compared on equal footing if you want a 4th variant."""
    ce_raw = F.cross_entropy(logits, targets, reduction='none')   # alpha NOT applied here
    pt = torch.exp(-ce_raw)                                        # true raw confidence
    loss = (1 - pt) ** gamma * ce_raw
    if alpha is not None:
        loss = loss * alpha[targets]                                # applied after, cleanly separate
    return loss.mean()


@torch.no_grad()
def evaluate_with_modality_holdout(model: nn.Module, X_pcap: np.ndarray, X_sar: np.ndarray,
                                    X_strace: np.ndarray, y: np.ndarray,
                                    hold_out: Optional[str] = None) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Evaluates an ALREADY-TRAINED model with one modality's input zeroed
    out (its own scaler's zero = "at the training-set mean", the natural
    neutral value in standardized space) - simulating that modality genuinely
    going missing at inference time (a dead sensor, a dropped capture), not
    retraining without it. `hold_out` in {"PCAP","SAR","STRACE",None}."""
    Xp = np.zeros_like(X_pcap) if hold_out == "PCAP" else X_pcap
    Xs = np.zeros_like(X_sar) if hold_out == "SAR" else X_sar
    Xt = np.zeros_like(X_strace) if hold_out == "STRACE" else X_strace
    loader = DataLoader(MultimodalDataset(Xp, Xs, Xt, y), BATCH_SIZE, shuffle=False)
    return evaluate_multimodal(model, loader)

def _safe_drop_last(n: int, shuffle: bool, batch_size: int = BATCH_SIZE) -> bool:
    """BatchNorm1d cannot compute batch statistics from a single sample, so a
    TRAINING loader whose last batch would have exactly 1 item crashes deep
    inside the model - this can happen on any split/shard whose size leaves a
    remainder of 1 against BATCH_SIZE (an FL client shard, an ablation
    subset, a small held-out slice, etc. - not specific to any one step).
    Only relevant for shuffled (i.e. training) loaders; eval-mode BatchNorm
    uses running statistics, so a size-1 eval batch is fine and dropping data
    from eval/test would be wrong. Only enabled when there's already more
    than one full batch, so we never drop an entire small dataset to zero."""
    return bool(shuffle and n > batch_size and n % batch_size == 1)

def train_epoch_multimodal(model, loader, optimizer, scaler_amp, class_weights, loss_fn=None):
    """`loss_fn(logits, y)` overrides the default class-weighted CE when
    given (used for the focal-loss weighting-scheme comparison) - lets that
    comparison reuse this exact training loop instead of a near-duplicate."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()
    for i, (xp, xs, xt, y) in enumerate(loader):
        xp, xs, xt, y = xp.to(DEVICE), xs.to(DEVICE), xt.to(DEVICE), y.to(DEVICE)
        with autocast(enabled=USE_AMP):
            logits = model(xp, xs, xt)
            if loss_fn is not None:
                loss = loss_fn(logits, y) / ACCUM_STEPS
            else:
                loss = (F.cross_entropy(logits, y, reduction='none') * class_weights[y]).mean() / ACCUM_STEPS
        scaler_amp.scale(loss).backward()
        if (i + 1) % ACCUM_STEPS == 0:
            scaler_amp.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_amp.step(optimizer); scaler_amp.update(); optimizer.zero_grad()
        total_loss += loss.item() * ACCUM_STEPS
        correct    += (logits.argmax(1) == y).sum().item()
        total      += len(y)
    return total_loss / max(len(loader), 1), correct / max(total, 1)

@torch.no_grad()
def evaluate_multimodal(model, loader, return_loss=False):
    model.eval()
    all_logits, all_y, total_loss = [], [], 0.0
    crit = nn.CrossEntropyLoss()
    for xp, xs, xt, y in loader:
        xp, xs, xt, y = xp.to(DEVICE), xs.to(DEVICE), xt.to(DEVICE), y.to(DEVICE)
        with autocast(enabled=USE_AMP):
            logits = model(xp, xs, xt)
            if return_loss:
                total_loss += crit(logits, y).item() * len(y)
        all_logits.append(logits.cpu())
        all_y.append(y.cpu())
    probs  = F.softmax(torch.cat(all_logits), dim=-1).numpy()
    preds  = probs.argmax(1)
    y_true = torch.cat(all_y).numpy()
    if return_loss:
        return preds, probs, y_true, total_loss / max(len(y_true), 1)
    return preds, probs, y_true

@torch.no_grad()
def evaluate_single(model, loader):
    model.eval()
    all_logits, all_y = [], []
    for xb, y in loader:
        with autocast(enabled=USE_AMP):
            all_logits.append(model(xb.to(DEVICE)).cpu())
        all_y.append(y)
    probs = F.softmax(torch.cat(all_logits), dim=-1).numpy()
    return probs.argmax(1), probs, torch.cat(all_y).numpy()

def print_metrics(y_true, y_pred, class_names, tag=""):
    acc    = accuracy_score(y_true, y_pred)
    mf1    = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    wf1    = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mf1_mi = f1_score(y_true, y_pred, average='micro',    zero_division=0)
    print(f"\n{'-'*60}")
    print(f"  {tag}")
    print(f"{'-'*60}")
    print(f"  Accuracy         : {acc*100:.2f}%")
    print(f"  Macro  F1        : {mf1:.4f}")
    print(f"  Weighted F1      : {wf1:.4f}")
    print(f"  Micro  F1        : {mf1_mi:.4f}")
    print(f"{'-'*60}")
    labels_present = sorted(set(y_true) | set(y_pred))
    names_present  = [class_names[i] for i in labels_present if i < len(class_names)]
    report = classification_report(
        y_true, y_pred, labels=labels_present, target_names=names_present, zero_division=0
    )
    print(report)
    return acc, mf1, wf1

In [ ]:
# ═══════════════════════════════════════════════════════════
# TREE-MODEL COMPLEXITY PROFILING + EXACT FL PAYLOAD SIZE
# ═══════════════════════════════════════════════════════════
def _tree_ensemble_size(model) -> Dict[str, Optional[float]]:
    """Structural complexity for tree ensembles. FLOPs are not a meaningful
    unit for a decision tree (prediction is a sequence of comparisons, not
    multiply-accumulates), so total node count and mean depth are reported in
    their place rather than leaving the row blank."""
    out = {"n_trees": None, "n_nodes": None, "mean_depth": None}
    if hasattr(model, "estimators_"):                       # RandomForest
        trees = model.estimators_
        out["n_trees"] = len(trees)
        out["n_nodes"] = int(sum(t.tree_.node_count for t in trees))
        out["mean_depth"] = float(np.mean([t.tree_.max_depth for t in trees]))
    else:                                                    # XGBoost
        try:
            df = model.get_booster().trees_to_dataframe()
            out["n_trees"] = int(df["Tree"].nunique())
            out["n_nodes"] = int(len(df))
            out["mean_depth"] = float(model.get_params().get("max_depth") or float("nan"))
        except Exception as e:
            print(f"[WARN] tree structure introspection failed: {e}")
    return out


def report_sklearn_complexity(model, X_sample: np.ndarray, name: str,
                              n_warmup: int = 10, n_runs: int = 100) -> Dict:
    """Wall-clock single-sample latency, serialized size and peak prediction
    memory for a fitted scikit-learn / XGBoost model, measured the same way as
    `report_model_complexity` does for the torch models: batch size 1, median
    over `n_runs` after `n_warmup` warmup calls."""
    import io, pickle, tracemalloc
    buf = io.BytesIO()
    pickle.dump(model, buf)
    size_mb = buf.tell() / (1024 ** 2)

    x1 = np.ascontiguousarray(X_sample[:1])
    for _ in range(n_warmup):
        model.predict(x1)

    tracemalloc.start()
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model.predict(x1)
        times.append((time.perf_counter() - t0) * 1000)
    peak_mb = tracemalloc.get_traced_memory()[1] / (1024 ** 2)
    tracemalloc.stop()

    struct = _tree_ensemble_size(model)
    report = {"model": name, "params": struct["n_nodes"], "flops": None,
              "size_mb": size_mb, "latency_ms_per_sample": float(np.median(times)),
              "peak_predict_mem_mb": peak_mb, **struct}
    print(f"[COMPLEXITY] {name:<20} nodes={struct['n_nodes'] or 0:>10,}  FLOPs=          n/a  "
          f"size={size_mb:>7.3f} MB  latency={np.median(times):>7.4f} ms/sample  "
          f"peak_mem={peak_mb:>6.3f} MB")
    return report


def state_payload_mb(model: nn.Module, exclude_keys=()) -> float:
    """Bytes actually put on the wire for one model transfer. Under FedBN the
    BatchNorm entries are never sent, so the true payload is smaller than the
    full `torch.save` size that Table 9 currently quotes."""
    sd = model.state_dict()
    total = sum(v.numel() * v.element_size() for k, v in sd.items() if k not in set(exclude_keys))
    return total / (1024 ** 2)

## Baselines

Shared training loop for flat models, and the Random Forest, XGBoost, MLP, and 1D-CNN baselines.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 15 — BASELINES (CLASSICAL + DEEP)
# ═══════════════════════════════════════════════════════════

def train_flat_model(model: nn.Module, X_tr_: np.ndarray, y_tr_: np.ndarray,
                      X_te_: np.ndarray, y_te_: np.ndarray, n_epochs: int,
                      class_weights: Optional[torch.Tensor], loss_fn=None,
                      augment: bool = True, tag: str = "", verbose: bool = True):
    """Generic training loop for any model taking ONE flattened feature
    tensor (SingleModalModel, MLPBaseline, TabularCNN). Pass `loss_fn(logits,
    targets)` to override the default class-weighted cross-entropy (used for
    the focal-loss comparison); pass `class_weights=None` with a custom
    `loss_fn` that doesn't need them (e.g. focal loss with no alpha)."""
    ds_tr = SingleModalDataset(X_tr_, y_tr_, augment=augment, noise_std=AUG_NOISE_STD)
    ld_tr = DataLoader(ds_tr, BATCH_SIZE, shuffle=True, drop_last=_safe_drop_last(len(ds_tr), True))
    ld_te = DataLoader(SingleModalDataset(X_te_, y_te_), BATCH_SIZE, shuffle=False)
    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, n_epochs))
    sc    = GradScaler(enabled=USE_AMP)
    cw    = class_weights.to(DEVICE) if class_weights is not None else None

    for _ in range(n_epochs):
        model.train()
        for xb, yb in ld_tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            with autocast(enabled=USE_AMP):
                logits = model(xb)
                loss = loss_fn(logits, yb) if loss_fn is not None else \
                       (F.cross_entropy(logits, yb, reduction='none') * cw[yb]).mean()
            opt.zero_grad()
            sc.scale(loss).backward(); sc.step(opt); sc.update()
        sched.step()

    yp, ypr, yt = evaluate_single(model, ld_te)
    if tag and verbose:
        print_metrics(yt, yp, class_names, f"{tag} ({n_epochs} epochs)")
    return yp, ypr, yt, model
    

def _metrics_from_preds(yt, yp) -> Dict:
    """macro_f1     : PRIMARY - mean F1 over classes with >= MIN_TEST_SUPPORT samples
                      in yt (all present classes if none qualify)
       macro_f1_all : mean F1 over every class present in yt
       per_class_*  : kept so any class subset can be recomputed without retraining"""
    labels   = np.unique(yt)
    f1_pc    = f1_score(yt, yp, labels=labels, average=None, zero_division=0)
    support  = np.array([(yt == l).sum() for l in labels])
    reliable = support >= MIN_TEST_SUPPORT
    return {
        'macro_f1':     float(f1_pc[reliable].mean() if reliable.any() else f1_pc.mean()),
        'macro_f1_all': float(f1_pc.mean()),
        'weighted_f1':  float(f1_score(yt, yp, labels=labels, average='weighted', zero_division=0)),
        'accuracy':     float(accuracy_score(yt, yp)),
        'per_class_f1':      {class_names[l]: float(f) for l, f in zip(labels, f1_pc)},
        'per_class_support': {class_names[l]: int(s)   for l, s in zip(labels, support)},
    }


def run_rf_baseline(X_tr, y_tr, X_te, y_te, class_names, seed: int = SEED, verbose: bool = True) -> Dict:
    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed).fit(X_tr, y_tr)
    yp = rf.predict(X_te)
    if verbose:
        acc, mf1, wf1 = print_metrics(y_te, yp, class_names, "Random Forest Baseline")
    return _metrics_from_preds(y_te, yp)

def _remap_labels_for_xgb(y_tr: np.ndarray) -> Tuple[np.ndarray, Dict[int, int]]:
    """XGBoost's multiclass objective requires TRAINING labels to be
    contiguous integers 0..k-1 - confirmed this applies even with no
    eval_set at all. Our GLOBAL label encoding is fixed across all classes
    (including ones excluded from training, e.g. via EXCLUDE_MALWARE_FAMILY,
    or a tiny merged class like 'Other' that randomly lands with zero
    training examples for a given seed - both are real, confirmed cases).
    Remapped from TRAINING classes only (not the union with test) - XGBoost
    additionally requires eval_set's classes to EXACTLY match training's,
    which a held-out class violates the moment it appears in test but not
    train, no matter how labels are remapped. We don't pass eval_set at all
    (it isn't used for anything here - no early stopping, no callbacks), so
    that constraint never applies; predictions always land within the
    training-derived local space, so the inverse map built from training
    classes alone is always sufficient to translate them back to the global
    label space."""
    present = np.unique(y_tr)
    local_of = {orig: i for i, orig in enumerate(present)}
    inverse  = {i: orig for orig, i in local_of.items()}
    y_tr_local = np.array([local_of[v] for v in y_tr])
    return y_tr_local, inverse


def run_xgb_baseline(X_tr, y_tr, X_te, y_te, class_names, seed: int = SEED, verbose: bool = True) -> Dict:
    if not HAS_XGB:
        return {'macro_f1': 0.0, 'macro_f1_all': 0.0, 'weighted_f1': 0.0, 'accuracy': 0.0,
                'per_class_f1': {}, 'per_class_support': {}}
    y_tr_local, inverse = _remap_labels_for_xgb(y_tr)
    clf = xgb.XGBClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=4,
        eval_metric='mlogloss', random_state=seed, tree_method='hist'
    )
    # No eval_set: see _remap_labels_for_xgb docstring - it isn't used for
    # anything here, and passing it re-triggers a class-set-match check that
    # a held-out class always fails regardless of remapping.
    yp_local = clf.fit(X_tr, y_tr_local).predict(X_te)
    yp = np.array([inverse[v] for v in yp_local])   # back to the GLOBAL label space
    if verbose:
        acc, mf1, wf1 = print_metrics(y_te, yp, class_names, "XGBoost Baseline")
    return _metrics_from_preds(y_te, yp)

def run_mlp_baseline(X_tr, y_tr, X_te, y_te, class_names, class_weights,
                      n_epochs: int = EPOCHS_DL, verbose: bool = True) -> Dict:
    model = MLPBaseline(X_tr.shape[1], N_CLASSES, hidden=HIDDEN_DIM * 2)
    yp, ypr, yt, _ = train_flat_model(model, X_tr, y_tr, X_te, y_te, n_epochs, class_weights,
                                       tag="MLP Baseline", verbose=verbose)
    return _metrics_from_preds(yt, yp)

def run_cnn_baseline(X_tr, y_tr, X_te, y_te, class_names, class_weights,
                      n_epochs: int = EPOCHS_DL, verbose: bool = True) -> Dict:
    model = TabularCNN(X_tr.shape[1], N_CLASSES)
    yp, ypr, yt, _ = train_flat_model(model, X_tr, y_tr, X_te, y_te, n_epochs, class_weights,
                                       tag="1D-CNN Baseline", verbose=verbose)
    return _metrics_from_preds(yt, yp)

## Plotting utilities

Training curves, confusion matrices, ROC/PR curves, UMAP projections with cluster scores, ablation and modality-holdout charts, and RL controller diagnostics. Figure data are also saved, so plots can be regenerated without retraining.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 16 — PLOTTING UTILITIES
# ═══════════════════════════════════════════════════════════
P = OUT_DIR / "plots"
UMAP_PARAMS = dict(n_neighbors=15, min_dist=0.1, metric='euclidean', n_components=2, random_state=SEED)

def _run_tag() -> str:
    """Encodes the active leave-one-out exclusion config into filenames so
    successive holdout runs (different EXCLUDE_ARCH / EXCLUDE_MALWARE_FAMILY
    settings) never silently overwrite each other's plots/reports."""
    parts = []
    if EXCLUDE_ARCH:
        parts.append(f"exArch-{EXCLUDE_ARCH}")
    if EXCLUDE_MALWARE_FAMILY:
        parts.append(f"exFam-{EXCLUDE_MALWARE_FAMILY}")
    return ("_" + "_".join(parts)) if parts else ""

def _savefig(name: str):
    plt.savefig(P / f"{name}{_run_tag()}.png", dpi=300, bbox_inches='tight')
    plt.close()

def compute_cluster_scores(embedding: np.ndarray, labels: np.ndarray) -> Dict[str, float]:
    """Quantitative cluster-separation scores for a 2D (or higher-D) embedding,
    computed against the true class labels. Requires >=2 classes with >=2 samples present in `labels`."""
    scores = {"silhouette": float("nan"), "davies_bouldin": float("nan"), "calinski_harabasz": float("nan")}
    uniq, counts = np.unique(labels, return_counts=True)
    if len(uniq) < 2 or (counts >= 2).sum() < 2:
        return scores
    try:
        scores["silhouette"] = float(silhouette_score(embedding, labels))
        scores["davies_bouldin"] = float(davies_bouldin_score(embedding, labels))
        scores["calinski_harabasz"] = float(calinski_harabasz_score(embedding, labels))
    except Exception as e:
        print(f"[WARN] cluster score computation failed: {e}")
    return scores

def plot_global_distribution(y, class_names):
    plt.figure(figsize=(8, 5))
    counts = pd.Series(y).value_counts(normalize=True).sort_index() * 100
    sns.barplot(x=counts.index, y=counts.values, palette="viridis")
    plt.xticks(ticks=range(len(class_names)), labels=class_names, rotation=45)
    plt.title('Class Distribution', fontweight='bold')
    plt.ylabel('Percentage (%)')
    _savefig("class_distribution")

def plot_client_distributions(client_y_dicts, class_names):
    for arch, y_vals in client_y_dicts.items():
        plt.figure(figsize=(6, 4))
        counts = pd.Series(y_vals).value_counts(normalize=True).sort_index() * 100
        sns.barplot(x=counts.index, y=counts.values, palette="magma")
        plt.xticks(ticks=range(len(class_names)), labels=class_names, rotation=45)
        plt.title(f'Class Distribution - {arch}', fontweight='bold')
        plt.ylabel('Percentage (%)')
        _savefig(f"class_dist_{arch.lower()}")

def plot_umap_raw_and_embedding(X_raw_dict, model, X_pcap, X_sar, X_strace, y, class_names,
                                 n_samples=2000) -> Dict[str, Dict[str, float]]:
    """Returns a dict of cluster scores keyed by embedding name, and also writes a text report with the UMAP 
    hyperparameters used (for reproducibility)."""
    if not HAS_UMAP: return {}
    all_scores = {}
    for mod, X in X_raw_dict.items():
        if X is None or X.shape[0] == 0 or X.shape[1] == 0: continue
        idx = np.random.choice(len(X), min(n_samples, len(X)), replace=False)
        emb = umap_lib.UMAP(**UMAP_PARAMS).fit_transform(X[idx])
        scores = compute_cluster_scores(emb, y[idx])
        all_scores[f"raw_{mod}"] = scores
        plt.figure(figsize=(8, 6))
        sc = plt.scatter(emb[:, 0], emb[:, 1], c=y[idx], cmap='tab10', s=SCATTER_POINT_SIZE, alpha=SCATTER_ALPHA, edgecolors='none')
        plt.legend(handles=sc.legend_elements()[0], labels=class_names, bbox_to_anchor=(1.05, 1), loc='upper right')
        plt.title(f"UMAP Features ({mod})\n"
                  f"silhouette={scores['silhouette']:.3f}  DB={scores['davies_bouldin']:.3f}  "
                  f"CH={scores['calinski_harabasz']:.1f}", fontsize=10)
        plt.grid(axis='x', linestyle='--', alpha=0.5, linewidth=0.5)
        _savefig(f"umap_raw_{mod.lower()}")

    model.eval()
    loader = DataLoader(MultimodalDataset(X_pcap, X_sar, X_strace, y), BATCH_SIZE, shuffle=False)
    embs = []
    for xp, xs, xt, _ in loader:
        with torch.no_grad():
            embs.append(model.get_embeddings(xp.to(DEVICE), xs.to(DEVICE), xt.to(DEVICE)).cpu().numpy())
    embs = np.concatenate(embs)
    idx  = np.random.choice(len(embs), min(n_samples, len(embs)), replace=False)
    emb_2d = umap_lib.UMAP(**UMAP_PARAMS).fit_transform(embs[idx])
    scores = compute_cluster_scores(emb_2d, y[idx])
    all_scores["learned_embedding"] = scores
    plt.figure(figsize=(8, 6))
    sc = plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=y[idx], cmap='tab10', s=SCATTER_POINT_SIZE, alpha=SCATTER_ALPHA, edgecolors='none')
    plt.legend(handles=sc.legend_elements()[0], labels=class_names, bbox_to_anchor=(1.05, 1), loc='upper right')
    plt.title(f"UMAP Learned Embeddings\n"
              f"silhouette={scores['silhouette']:.3f}  DB={scores['davies_bouldin']:.3f}  "
              f"CH={scores['calinski_harabasz']:.1f}", fontsize=10)
    _savefig("umap_embeddings")

    with open(OUT_DIR / "reports" / f"umap_cluster_scores_{EXCLUDE_ARCH}.json", "w") as f:
        json.dump({"umap_params": UMAP_PARAMS, "scores": all_scores}, f, indent=2)
    print("\n[UMAP] Cluster separation scores (higher silhouette/CH = better, lower DB = better):")
    for name, s in all_scores.items():
        print(f"  {name:<18} silhouette={s['silhouette']:.4f}  davies_bouldin={s['davies_bouldin']:.4f}  "
              f"calinski_harabasz={s['calinski_harabasz']:.2f}")
    print(f"  UMAP params used: {UMAP_PARAMS}")
    return all_scores

def plot_correlation_matrices(X_dict):
    for mod, X in X_dict.items():
        if X is None or X.shape[0] == 0 or X.shape[1] == 0: continue
        corr = pd.DataFrame(X[:1000, :min(40, X.shape[1])]).corr()
        plt.figure(figsize=(8, 6))
        sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1,
                    xticklabels=False, yticklabels=False, square=True)
        plt.title(f'Feature Correlation ({mod})', fontweight='bold')
        _savefig(f"corr_{mod.lower()}")

def plot_training_curves(histories: List[Dict]):
    """Accepts a LIST of per-seed history dicts (length 1 is fine for a
    single run). Plots the per-epoch MEAN as a line with a shaded +/-1 std
    band across seeds (the x-axis is the epoch count; the title states how
    many seeds went in)."""
    n_epochs = len(histories[0]['train_loss'])
    eps = np.arange(1, n_epochs + 1)
    def stack(key): return np.array([h[key][:n_epochs] for h in histories])  # (n_seeds, n_epochs)

    for key_pair, title, fname, ylabel in [(('train_loss', 'val_loss'), 'Loss', 'central_loss', 'Loss'),
                                    (('train_acc', 'val_acc'), 'Accuracy', 'central_accuracy', 'Accuracy')]:
        plt.figure(figsize=(5, 4))
        for key, color, style, label in [(key_pair[0], 'tab:blue', '-', 'Train'),
                                          (key_pair[1], 'tab:red', '--', 'Validation')]:
            arr = stack(key)
            mean, std = arr.mean(axis=0), arr.std(axis=0)
            plt.plot(eps, mean, style, color=color, label=label, lw=1)
            plt.fill_between(eps, mean - std, mean + std, color=color, alpha=0.15)
            plt.ylabel(f'{ylabel}')
        plt.title(f'{title}')
        plt.xlabel('Epoch'); plt.legend()
        plt.grid(axis='x', linestyle='--', alpha=0.5, linewidth=0.5)
        _savefig(fname)

def plot_confusion_matrix(y_true, y_pred, class_names, prefix):
    # Explicit labels=, same reasoning as plot_per_class_radar: without it, a
    # class absent from both y_true and y_pred shrinks the matrix below
    # len(class_names) and silently mislabels the axes instead of erroring.
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.grid(axis='x', linestyle='--', alpha=0.5, linewidth=0.5)
    _savefig(f"cm_{prefix.lower()}")

def plot_pr_curves(y_true, y_prob, class_names, fname):
    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    plt.figure(figsize=(9, 6))
    for i, name in enumerate(class_names):
        if y_bin[:, i].sum() == 0: continue
        p, r, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
        ap = average_precision_score(y_bin[:, i], y_prob[:, i])
        plt.plot(r, p, lw=1, label=f'{name} (AP={ap:.2f})')
    plt.xlabel('Recall'); plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(fontsize=8)
    _savefig(fname)

def plot_roc_curves(y_true, y_prob, class_names, fname):
    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    plt.figure(figsize=(9, 6))
    for i, name in enumerate(class_names):
        if y_bin[:, i].sum() == 0: continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        plt.plot(fpr, tpr, lw=1, label=f'{name} (AUC={auc(fpr,tpr):.2f})')
    plt.plot([0,1],[0,1],'k--')
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title('ROC Curve')
    plt.legend(fontsize=7, loc='lower right'); plt.tight_layout()
    _savefig(fname)

def plot_confidence_metrics(y_true, y_pred, y_prob, class_names, prefix="central"):
    confidences = np.max(y_prob, axis=1)
    plt.figure(figsize=(8, 5))
    plt.hist(confidences[y_true == y_pred], bins=20, alpha=0.6, color='green', density=True, label='Correct')
    plt.hist(confidences[y_true != y_pred], bins=20, alpha=0.6, color='red',   density=True, label='Incorrect')
    plt.title('Prediction Confidence', fontweight='bold'); plt.ylabel('Density'); plt.legend()
    _savefig(f"confidence_hist_{prefix}")

    df = pd.DataFrame([{'True Class': class_names[y_true[i]], 'Prob': y_prob[i, y_true[i]]} for i in range(len(y_true))])
    plt.figure(figsize=(10, 5))
    sns.violinplot(x='True Class', y='Prob', data=df, palette='muted', inner='quartile')
    plt.xticks(rotation=45); plt.title('Probability per Class', fontweight='bold'); plt.ylim(0, 1.1)
    _savefig(f"violin_probs_{prefix}")

def plot_fl_global(fl_histories: List[Dict]):
    """Same mean +/- std treatment as plot_training_curves, across FL
    rounds instead of epochs. 'Global' here is the size-weighted mean of
    each round's personalized clients (see run_fl_seed) - not a single
    blended-BN model, since FedBN has no such thing to evaluate."""
    n_rounds = len(fl_histories[0]['global_acc'])
    rounds = np.arange(1, n_rounds + 1)
    def stack(key): return np.array([h[key][:n_rounds] for h in fl_histories])
    n_seeds = len(fl_histories)

    for key, title, fname, color, ylabel in [('global_loss', 'Federated Learning Global Loss', 'fl_global_loss', 'tab:red', 'Loss'),
                                      ('global_acc', 'Federated Learning Global Accuracy', 'fl_global_accuracy', 'tab:blue', 'Accuracy'),
                                      ('global_f1', 'Federated Learning Global Macro-F1', 'fl_global_f1', 'tab:green', 'Macro F1')]:
        arr = stack(key)
        mean, std = arr.mean(axis=0), arr.std(axis=0)
        plt.figure(figsize=(5, 4))
        plt.plot(rounds, mean, '-o', color=color, markersize=2, lw=1)
        plt.fill_between(rounds, mean - std, mean + std, color=color, alpha=0.15)
        plt.title(f'{title}')
        plt.ylabel(f'{ylabel}')
        plt.xlabel('Round')
        plt.grid(axis='x', linestyle='--', alpha=0.5, linewidth=0.5)
        _savefig(fname)

def plot_fl_client_heatmap(client_f1_history):
    df = pd.DataFrame(client_f1_history).T
    df.columns = [f"R{c+1}" for c in df.columns]
    plt.figure(figsize=(12, max(4, len(df)*0.8)))
    sns.heatmap(df, cmap='YlGnBu', annot=True, fmt='.3f', cbar_kws={'label': 'F1-Macro'})
    plt.title('Client F1 Across FL Rounds (true per-architecture clients)', fontweight='bold')
    plt.xlabel('Round'); plt.ylabel('Architecture'); plt.tight_layout()
    _savefig("fl_heatmap")

def plot_rl_dynamics(reward_history, weight_history, class_names, phase="Central"):
    if reward_history:
        ma = pd.Series(reward_history).rolling(max(2, len(reward_history)//10), min_periods=1).mean()
        plt.figure(figsize=(5, 4))
        plt.plot(reward_history, alpha=0.4, color='blue', label='Raw d(recall signal)')
        plt.plot(ma.values, color='red', lw=1, label='Moving Avg')
        plt.axhline(0, color='black', linestyle='--')
        plt.title('RL Reward')
        plt.xlabel('Step'); plt.legend()
        _savefig(f"rl_rewards_{phase.lower()}")
    if weight_history:
        wh = np.array(weight_history)
        plt.figure(figsize=(5, 4))
        plt.stackplot(range(wh.shape[0]), wh.T, labels=class_names, alpha=0.7, cmap='tab10')
        plt.title('Class Weight Evolution')
        plt.xlabel('Step'); plt.ylabel('Weight')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); plt.tight_layout()
        _savefig(f"rl_weights_{phase.lower()}")

def plot_modality_holdout(holdout_results: Dict[str, Tuple[float, float]], full_f1: Tuple[float, float]):
    """Bar chart of macro-F1 when one modality is zeroed out at test time vs.
    the full-modality baseline (partial inference with a missing modality)."""
    names = ['Complete'] + [f'Missing {m}' for m in holdout_results]
    means = [full_f1[0]] + [holdout_results[m][0] for m in holdout_results]
    stds  = [full_f1[1]] + [holdout_results[m][1] for m in holdout_results]
    colors = ['#2ca02c'] + ['#d62728'] * len(holdout_results)
    plt.figure(figsize=(5, 4))
    bars = plt.bar(names, means, yerr=stds, color=colors, capsize=3, edgecolor='black', lw=0.4)
    for bar, m, s in zip(bars, means, stds):
        plt.text(bar.get_x() + bar.get_width()/2, m + s + 0.01, f'{m:.3f}', ha='center', fontsize=8)
    plt.ylabel('Macro-F1'); plt.ylim(0, 1.05)
    plt.title(f'Modality Holdout (mean±std)')
    plt.xticks(rotation=15, ha='right')
    plt.grid(axis='x', linestyle='--', alpha=0.5, linewidth=0.5)
    plt.tight_layout()
    _savefig("modality_holdout")

def plot_ablation_radar(ablation_results: Dict[str, Tuple[float, float]], all_model_results: Dict):
    """`ablation_results` values are now (mean, std) tuples across seeds,
    not single floats - error bars show training-run variance directly on
    the bar chart."""
    names  = list(ablation_results.keys())
    means  = [ablation_results[n][0] for n in names]
    stds   = [ablation_results[n][1] for n in names]
    colors = ['#d62728' if 'Baseline' in n else '#1f77b4' if 'only' in n
              else '#ff7f0e' if '+' in n else '#2ca02c' for n in names]
    plt.figure(figsize=(12, max(4, len(names) * 0.4)))
    bars = plt.barh(names, means, xerr=stds, color=colors, edgecolor='black', lw=0.4,
                     error_kw=dict(elinewidth=1, capsize=3))
    if means: plt.axvline(max(means), color='red', lw=0.8, linestyle='--')
    for bar, m, s in zip(bars, means, stds):
        plt.text(m + s + 0.005, bar.get_y() + bar.get_height() / 2, f'{m:.3f}±{s:.3f}', va='center', fontsize=7)
    n_seeds = len(SEEDS)
    plt.xlabel('Macro-F1')
    plt.title(f'Ablation Study (mean±std)')
    plt.tight_layout()
    _savefig("ablation_bar")

    mods = ['PCAP only', 'SAR only', 'STRACE only']
    if all(m in ablation_results for m in mods):
        s = [ablation_results[m][0] for m in mods]
        angles = [n / 3 * 2 * pi for n in range(3)]
        s += s[:1]; angles += angles[:1]
        fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
        ax.plot(angles, s, linewidth=2, color='darkorange'); ax.fill(angles, s, alpha=0.25,color='orange')
        ax.set_xticks(angles[:-1]); ax.set_xticklabels(['PCAP', 'SAR', 'STRACE'])
        plt.title('Single Modality Contribution (mean macro-F1)')
        _savefig("radar_modalities")

    if all_model_results:
        models = list(all_model_results.keys())
        macro  = [all_model_results[m].get('macro_f1', 0) for m in models]
        wt     = [all_model_results[m].get('weighted_f1', 0) for m in models]
        macro_std = [all_model_results[m].get('macro_f1_std', 0) for m in models]
        wt_std    = [all_model_results[m].get('weighted_f1_std', 0) for m in models]
        x = np.arange(len(models))
        plt.figure(figsize=(max(8, len(models) * 1.2), 5))
        plt.bar(x - 0.2, macro, 0.4, yerr=macro_std, label='Macro-F1',    color='royalblue', capsize=3)
        plt.bar(x + 0.2, wt,    0.4, yerr=wt_std,    label='Weighted-F1', color='tomato', capsize=3)
        plt.xticks(x, models, rotation=30, ha='right', fontsize=8)
        plt.ylabel('Score'); plt.ylim(0, 1.05)
        plt.title(f'Macro vs Weighted F1 (mean±std)')
        plt.legend(); plt.tight_layout()
        _savefig("macro_vs_weighted")

def plot_per_class_radar(y_true, y_pred, class_names, phase):
    # Explicit `labels=` (not just target_names) matters here: if a class has
    # zero samples in both y_true AND y_pred (e.g. it was excluded via
    # EXCLUDE_MALWARE_FAMILY and never appears in this test set),
    # classification_report's default label-inference from y_true/y_pred
    # would silently drop it, leaving fewer labels than target_names and
    # raising a ValueError instead of just plotting it as 0.
    labels = list(range(len(class_names)))
    report = classification_report(y_true, y_pred, labels=labels, target_names=class_names,
                                    output_dict=True, zero_division=0)
    scores = [report.get(c, {}).get('f1-score', 0) for c in class_names]
    angles = [n / float(len(class_names)) * 2 * pi for n in range(len(class_names))]
    scores += scores[:1]; angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    ax.plot(angles, scores, linewidth=2, color='purple')
    ax.fill(angles, scores, alpha=0.25, color='purple')
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(class_names)
    plt.ylim(0, 1.0); plt.title('Per-Class F1-Score')
    _savefig(f"per_class_radar_{phase.lower()}")

def plot_central_vs_fl(histories_central: List[Dict], fl_histories: List[Dict]):
    """Accepts LISTS of per-seed histories now, matching plot_training_curves
    / plot_fl_global - plots each side's mean across seeds (epochs vs.
    rounds are different x-axes, so this stays a simple mean line rather
    than adding error bands on top of an already dual-axis comparison)."""
    for metric, ylabel in [('acc', 'Accuracy'), ('f1', 'Macro-F1')]:
        c_arr = np.array([h[f'val_{metric}'] for h in histories_central])
        f_arr = np.array([h[f'global_{metric}'] for h in fl_histories])
        plt.figure(figsize=(5, 4))
        plt.plot(
            range(1, c_arr.shape[1] + 1),
            c_arr.mean(axis=0),
            color='indigo',
            linestyle='-',
            label=ROW_DRL
        )
        
        plt.plot(
            range(1, f_arr.shape[1] + 1),
            f_arr.mean(axis=0),
            color='crimson',
            linestyle='--',
            label=ROW_FL
        )
        plt.title(f'{ylabel}')
        plt.ylabel(f'{ylabel}')
        plt.xlabel('Epochs / Rounds'); plt.legend()
        _savefig(f"central_vs_fl_{metric}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# FIGURE STYLE + REPLOT-WITHOUT-RETRAIN ARTIFACTS
# ═══════════════════════════════════════════════════════════
plt.rcParams.update({
    "font.size":        BASE_FONT_PT,
    "axes.titlesize":   BASE_FONT_PT + 1,
    "axes.labelsize":   BASE_FONT_PT,
    "xtick.labelsize":  BASE_FONT_PT - 1,
    "ytick.labelsize":  BASE_FONT_PT - 1,
    "legend.fontsize":  BASE_FONT_PT - 1,
    "legend.title_fontsize": BASE_FONT_PT - 1,
    "figure.dpi":       300,
    "savefig.dpi":      300,
    "lines.linewidth":  1.1,
    "axes.prop_cycle":  plt.cycler(color=CLASS_PALETTE),
})


def _class_style(i: int):
    """(colour, linestyle) for class i — hue and dash pattern both vary, so the
    curves stay separable in greyscale and under colour-vision deficiency."""
    return (CLASS_PALETTE[i % len(CLASS_PALETTE)],
            CLASS_LINESTYLES[i % len(CLASS_LINESTYLES)])


def save_eval_artifacts(name: str, y_true, y_pred, y_prob, class_names) -> Path:
    """Persists everything Fig. 12 and the confusion matrices are drawn from,
    so they can be redrawn later without retraining a single model.

    Writes two files per config:
      <name>.npz          raw y_true / y_pred / y_prob  (canonical source)
      <name>_curves.json  per-class ROC and PR points, AUC, AP, support
    """
    if not SAVE_EVAL_ARTIFACTS:
        return None
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)
    npz_path = ARTIFACT_DIR / f"{name}{_run_tag()}.npz"
    np.savez_compressed(npz_path, y_true=y_true, y_pred=y_pred, y_prob=y_prob,
                        class_names=np.asarray(class_names, dtype=object))

    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    curves = {}
    for i, cname in enumerate(class_names):
        if y_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        prec, rec, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
        curves[cname] = {
            "n_positive": int(y_bin[:, i].sum()),
            "auc": float(auc(fpr, tpr)),
            "average_precision": float(average_precision_score(y_bin[:, i], y_prob[:, i])),
            "roc": {"fpr": fpr.tolist(), "tpr": tpr.tolist()},
            "pr":  {"recall": rec.tolist(), "precision": prec.tolist()},
        }
    with open(ARTIFACT_DIR / f"{name}{_run_tag()}_curves.json", "w") as f:
        json.dump({"config": name, "run_tag": _run_tag(),
                   "classes": list(class_names), "curves": curves}, f)
    print(f"[ARTIFACT] {name}: {npz_path.name} + {name}{_run_tag()}_curves.json")
    return npz_path


def load_eval_artifacts(name: str, run_tag: Optional[str] = None) -> Dict:
    tag = _run_tag() if run_tag is None else run_tag
    d = np.load(ARTIFACT_DIR / f"{name}{tag}.npz", allow_pickle=True)
    return {"y_true": d["y_true"], "y_pred": d["y_pred"], "y_prob": d["y_prob"],
            "class_names": [str(c) for c in d["class_names"]]}


def replot_curves_from_artifacts(name: str, run_tag: Optional[str] = None):
    """Redraws ROC, PR and the confusion matrix for a saved config. No model,
    no training, no GPU — works from a fresh kernel with only the zip restored."""
    a = load_eval_artifacts(name, run_tag)
    plot_roc_curves(a["y_true"], a["y_prob"], a["class_names"], f"{name}_roc")
    plot_pr_curves(a["y_true"], a["y_prob"], a["class_names"], f"{name}_pr")
    plot_confusion_matrix(a["y_true"], a["y_pred"], a["class_names"], name)
    print(f"[REPLOT] {name}: roc, pr, confusion matrix regenerated from artifacts")


def plot_roc_curves(y_true, y_prob, class_names, fname):
    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    plt.figure(figsize=PANEL_FIGSIZE)
    ax = plt.gca()
    for i, name in enumerate(class_names):
        if y_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        colour, ls = _class_style(i)
        ax.plot(fpr, tpr, lw=1.1, color=colour, linestyle=ls,
                label=f'{name} ({auc(fpr, tpr):.2f})')
    ax.plot([0, 1], [0, 1], color='0.6', lw=0.7, linestyle=(0, (1, 2)))
    ax.set_xlabel('False positive rate')
    ax.set_ylabel('True positive rate')
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
    ax.legend(loc='lower right', frameon=True, framealpha=0.9,
              handlelength=1.8, borderpad=0.3, labelspacing=0.25,
              title='Class (AUC)')
    plt.tight_layout(pad=0.3)
    _savefig(fname)


def plot_pr_curves(y_true, y_prob, class_names, fname):
    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    plt.figure(figsize=PANEL_FIGSIZE)
    ax = plt.gca()
    for i, name in enumerate(class_names):
        if y_bin[:, i].sum() == 0:
            continue
        prec, rec, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
        ap = average_precision_score(y_bin[:, i], y_prob[:, i])
        colour, ls = _class_style(i)
        ax.plot(rec, prec, lw=1.1, color=colour, linestyle=ls,
                label=f'{name} ({ap:.2f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
    ax.legend(loc='upper right', frameon=True, framealpha=0.9,
              handlelength=1.8, borderpad=0.3, labelspacing=0.25,
              title='Class (AP)')
    plt.tight_layout(pad=0.3)
    _savefig(fname)


def plot_confusion_matrix(y_true, y_pred, class_names, prefix):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=PANEL_FIGSIZE)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True,
                cbar=False, linewidths=0.3, linecolor='white',
                annot_kws={"size": BASE_FONT_PT - 1},
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
    plt.tight_layout(pad=0.3)
    _savefig(f"cm_{prefix.lower()}")


def plot_rl_dynamics(reward_history, weight_history, class_names, phase="Central"):
    """`stackplot` takes `colors=`, not `cmap=` — the previous `cmap='tab10'`
    was silently dropped and the default cycle used instead, which is exactly
    which produced similar-toned segments."""
    if reward_history:
        ma = pd.Series(reward_history).rolling(max(2, len(reward_history) // 10),
                                               min_periods=1).mean()
        plt.figure(figsize=PANEL_FIGSIZE)
        plt.plot(reward_history, alpha=0.35, color=CLASS_PALETTE[0], lw=0.8, label='Per-step')
        plt.plot(ma.values, color=CLASS_PALETTE[1], lw=1.3, label='Moving average')
        plt.axhline(0, color='0.4', linestyle='--', lw=0.7)
        plt.xlabel('Step'); plt.ylabel('Reward')
        plt.legend(frameon=True, framealpha=0.9, handlelength=1.8, borderpad=0.3)
        plt.tight_layout(pad=0.3)
        _savefig(f"rl_rewards_{phase.lower()}")

    if weight_history:
        wh = np.array(weight_history)
        n_cls = wh.shape[1]
        colours = [CLASS_PALETTE[i % len(CLASS_PALETTE)] for i in range(n_cls)]
        plt.figure(figsize=WIDE_FIGSIZE)
        plt.stackplot(range(wh.shape[0]), wh.T, labels=class_names,
                      colors=colours, alpha=0.85,
                      edgecolor='white', linewidth=0.4)
        plt.xlabel('Step'); plt.ylabel('Class weight')
        plt.xlim(0, wh.shape[0] - 1)
        plt.legend(bbox_to_anchor=(1.01, 1.0), loc='upper left',
                   frameon=False, handlelength=1.2, labelspacing=0.3)
        plt.tight_layout(pad=0.3)
        _savefig(f"rl_weights_{phase.lower()}")


def _umap_panel(emb, lab, scores, title, fname, class_names):
    """Draws one UMAP panel. Shared by the live plot and the replot, so a
    redrawn figure is pixel-for-pixel the same layout as the original."""
    from matplotlib.colors import ListedColormap
    from matplotlib.lines import Line2D
    cmap = ListedColormap(CLASS_PALETTE[:len(class_names)])
    handles = [Line2D([], [], marker='o', linestyle='', markersize=3,
                      color=CLASS_PALETTE[i % len(CLASS_PALETTE)], label=c)
               for i, c in enumerate(class_names)]
    plt.figure(figsize=PANEL_FIGSIZE)
    ax = plt.gca()
    ax.scatter(emb[:, 0], emb[:, 1], c=lab, cmap=cmap,
               vmin=-0.5, vmax=len(class_names) - 0.5,
               s=SCATTER_POINT_SIZE, alpha=SCATTER_ALPHA, edgecolors='none')
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.02, 0.02,
            f"sil {scores['silhouette']:.3f}\nDB {scores['davies_bouldin']:.3f}\n"
            f"CH {scores['calinski_harabasz']:.0f}",
            transform=ax.transAxes, va='bottom', ha='left',
            fontsize=BASE_FONT_PT - 1,
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                      edgecolor='0.7', alpha=0.85))
    ax.legend(handles=handles, loc='upper right', frameon=True, framealpha=0.85,
              handlelength=1.0, borderpad=0.25, labelspacing=0.2,
              fontsize=BASE_FONT_PT - 2)
    plt.tight_layout(pad=0.3)
    _savefig(fname)


def plot_umap_raw_and_embedding(X_raw_dict, model, X_pcap, X_sar, X_strace, y, class_names,
                                n_samples=2000) -> Dict[str, Dict[str, float]]:
    """Plots the three raw-modality UMAPs and the fused learned embedding, and
    saves every panel's 2-D coordinates and labels so the figures can be
    redrawn without the model."""
    if not HAS_UMAP:
        return {}
    all_scores, panels, store = {}, [], {}

    def _record(emb, lab, scores, title, fname, score_key):
        all_scores[score_key] = scores
        _umap_panel(emb, lab, scores, title, fname, class_names)
        store[f"{fname}__emb"] = np.asarray(emb, dtype=np.float32)
        store[f"{fname}__lab"] = np.asarray(lab)
        panels.append({"fname": fname, "title": title, "score_key": score_key})

    for mod, X in X_raw_dict.items():
        if X is None or X.shape[0] == 0 or X.shape[1] == 0:
            continue
        idx = np.random.choice(len(X), min(n_samples, len(X)), replace=False)
        emb = umap_lib.UMAP(**UMAP_PARAMS).fit_transform(X[idx])
        _record(emb, y[idx], compute_cluster_scores(emb, y[idx]),
                f"Raw {mod} features", f"umap_raw_{mod.lower()}", f"raw_{mod}")

    model.eval()
    loader = DataLoader(MultimodalDataset(X_pcap, X_sar, X_strace, y), BATCH_SIZE, shuffle=False)
    embs = []
    for xp, xs, xt, _ in loader:
        with torch.no_grad():
            embs.append(model.get_embeddings(xp.to(DEVICE), xs.to(DEVICE), xt.to(DEVICE)).cpu().numpy())
    embs = np.concatenate(embs)
    idx = np.random.choice(len(embs), min(n_samples, len(embs)), replace=False)
    emb_2d = umap_lib.UMAP(**UMAP_PARAMS).fit_transform(embs[idx])
    _record(emb_2d, y[idx], compute_cluster_scores(emb_2d, y[idx]),
            "Fused learned embedding", "umap_embeddings", "learned_embedding")

    np.savez_compressed(ARTIFACT_DIR / f"umap_panels{_run_tag()}.npz", **store)
    with open(OUT_DIR / "reports" / f"umap_cluster_scores{_run_tag()}.json", "w") as f:
        json.dump({"umap_params": {k: str(v) for k, v in UMAP_PARAMS.items()},
                   "scores": all_scores, "panels": panels}, f, indent=2, default=float)
    print(f"[ARTIFACT] UMAP coordinates for {len(panels)} panels -> umap_panels{_run_tag()}.npz")

    print("\n[UMAP] Cluster separation scores:")
    for name, s in all_scores.items():
        print(f"  {name:<18} silhouette={s['silhouette']:.4f}  "
              f"davies_bouldin={s['davies_bouldin']:.4f}  "
              f"calinski_harabasz={s['calinski_harabasz']:.2f}")
    return all_scores


def replot_umap_from_artifacts(class_names, run_tag: Optional[str] = None):
    tag = _run_tag() if run_tag is None else run_tag
    d = np.load(ARTIFACT_DIR / f"umap_panels{tag}.npz")
    with open(OUT_DIR / "reports" / f"umap_cluster_scores{tag}.json") as f:
        meta = json.load(f)
    for p in meta["panels"]:
        _umap_panel(d[f"{p['fname']}__emb"], d[f"{p['fname']}__lab"],
                    meta["scores"][p["score_key"]], p["title"], p["fname"], class_names)
    print(f"[REPLOT] {len(meta['panels'])} UMAP panels")


def plot_ablation_radar(ablation_results: Dict[str, Tuple[float, float]], all_model_results: Dict):
    """Bar labels were 7 pt on a 12 in figure printed at ~6.5 in — sub-4 pt on
    the page. Height now scales with the config count at a fixed
    print width, and the value labels sit at the base font size."""
    names = list(ablation_results.keys())
    means = [ablation_results[n][0] for n in names]
    stds  = [ablation_results[n][1] for n in names]
    order = np.argsort(means)
    names = [names[i] for i in order]
    means = [means[i] for i in order]
    stds  = [stds[i] for i in order]

    _CATEGORY_COLOUR = {"proposed": CLASS_PALETTE[0], "weighting": CLASS_PALETTE[1],
                        "modality": CLASS_PALETTE[2], "baseline": '0.6'}

    def _colour(n):
        return _CATEGORY_COLOUR.get(ROW_CATEGORY.get(n, "baseline"), '0.6')

    plt.figure(figsize=(WIDE_FIGSIZE[0], max(3.0, len(names) * 0.28)))
    bars = plt.barh(names, means, xerr=stds, color=[_colour(n) for n in names],
                    edgecolor='black', lw=0.4, height=0.7,
                    error_kw=dict(elinewidth=0.8, capsize=2))
    for bar, m, s in zip(bars, means, stds):
        plt.text(m + s + 0.012, bar.get_y() + bar.get_height() / 2,
                 f'{m:.3f}', va='center', fontsize=BASE_FONT_PT - 1)
    plt.xlabel('Macro-F1')
    if means:
        plt.xlim(0, min(1.0, max(m + s for m, s in zip(means, stds)) + 0.10))
    plt.tight_layout(pad=0.3)
    _savefig("ablation_bar")

    mods = [ROW_SINGLE[m] for m in MODALITIES]
    if all(m in ablation_results for m in mods):
        s = [ablation_results[m][0] for m in mods]
        angles = [n / 3 * 2 * pi for n in range(3)]
        s += s[:1]; angles += angles[:1]
        fig, ax = plt.subplots(figsize=PANEL_FIGSIZE, subplot_kw=dict(polar=True))
        ax.plot(angles, s, linewidth=1.4, color=CLASS_PALETTE[1])
        ax.fill(angles, s, alpha=0.25, color=CLASS_PALETTE[1])
        ax.set_xticks(angles[:-1]); ax.set_xticklabels(['PCAP', 'SAR', 'STRACE'])
        ax.tick_params(labelsize=BASE_FONT_PT - 1)
        plt.tight_layout(pad=0.3)
        _savefig("radar_modalities")

    if all_model_results:
        models = list(all_model_results.keys())
        macro = [all_model_results[m].get('macro_f1', 0) for m in models]
        wt    = [all_model_results[m].get('weighted_f1', 0) for m in models]
        macro_std = [all_model_results[m].get('macro_f1_std', 0) for m in models]
        wt_std    = [all_model_results[m].get('weighted_f1_std', 0) for m in models]
        x = np.arange(len(models))
        plt.figure(figsize=(WIDE_FIGSIZE[0], 3.4))
        plt.bar(x - 0.2, macro, 0.4, yerr=macro_std, label='Macro-F1',
                color=CLASS_PALETTE[0], capsize=2, error_kw=dict(elinewidth=0.8))
        plt.bar(x + 0.2, wt, 0.4, yerr=wt_std, label='Weighted-F1',
                color=CLASS_PALETTE[1], capsize=2, error_kw=dict(elinewidth=0.8))
        plt.xticks(x, models, rotation=35, ha='right')
        plt.ylabel('Score'); plt.ylim(0, 1.05)
        plt.legend(frameon=False)
        plt.tight_layout(pad=0.3)
        _savefig("macro_vs_weighted")

def plot_policy_diagnostics(agent, phase="Central"):
    """Three diagnostics of whether the controller learned a policy:
      alpha_0   — concentration. Rising alpha_0 means the policy is becoming
                  confident; flat-and-low means it never committed.
      H[pi]     — policy entropy. Monotone collapse to a corner would falsify
                  the claim that the entropy term regularises exploration.
      ||a-mu||1 — how far the sampled action sits from the mean action, i.e.
                  how much exploration actually happened."""
    if not agent.alpha0_history:
        return
    steps = np.arange(1, len(agent.alpha0_history) + 1)
    fig, axes = plt.subplots(3, 1, figsize=(PANEL_FIGSIZE[0], 4.6), sharex=True)
    for ax, series, label, colour in zip(
            axes,
            [agent.alpha0_history, agent.entropy_history, agent.explore_gap_history],
            [r'$\alpha_0$', r'$H[\pi]$', r'$\|a-\mu\|_1$'],
            CLASS_PALETTE[:3]):
        ax.plot(steps, series, color=colour, lw=1.0)
        ax.set_ylabel(label)
        ax.grid(alpha=0.3, lw=0.4)
    axes[-1].set_xlabel('Policy update step')
    plt.tight_layout(pad=0.3)
    _savefig(f"policy_diagnostics_{phase.lower()}")

## Build the splits

Creates the splits, fits the pre-processing on the training split, and builds the data loaders. val_reward is used only by the RL controller; val_select is used for hyperparameter selection and monitoring. Every model is evaluated at its final epoch.

In [ ]:
bundle = build_model_ready_splits(full_df)
N_CLASSES   = bundle["n_classes"]
class_names = bundle["class_names"]
idx_tr, idx_te = bundle["idx_tr"], bundle["idx_te"]
idx_val_select, idx_val_reward = bundle["idx_val_select"], bundle["idx_val_reward"]

X_pcap_tr,  X_sar_tr,  X_strace_tr  = bundle["X"]["train"]["PCAP"], bundle["X"]["train"]["SAR"], bundle["X"]["train"]["STRACE"]
X_pcap_vs,  X_sar_vs,  X_strace_vs  = bundle["X"]["val_select"]["PCAP"], bundle["X"]["val_select"]["SAR"], bundle["X"]["val_select"]["STRACE"]
X_pcap_vr,  X_sar_vr,  X_strace_vr  = bundle["X"]["val_reward"]["PCAP"], bundle["X"]["val_reward"]["SAR"], bundle["X"]["val_reward"]["STRACE"]
X_pcap_te,  X_sar_te,  X_strace_te  = bundle["X"]["test"]["PCAP"],  bundle["X"]["test"]["SAR"],  bundle["X"]["test"]["STRACE"]
y_tr, y_val_select, y_val_reward, y_te = (bundle["y"]["train"], bundle["y"]["val_select"],
                                           bundle["y"]["val_reward"], bundle["y"]["test"])
idx_te_is_excluded = bundle["idx_te_is_excluded"]

dim_pcap, dim_sar, dim_strace = bundle["dims"]["PCAP"], bundle["dims"]["SAR"], bundle["dims"]["STRACE"]

def make_loader(Xp, Xs, Xt, y, shuffle, augment=False):
    ds = MultimodalDataset(Xp, Xs, Xt, y, augment=augment, noise_std=AUG_NOISE_STD)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=_safe_drop_last(len(ds), shuffle))

train_loader      = make_loader(X_pcap_tr, X_sar_tr, X_strace_tr, y_tr, True, USE_AUGMENTATION)
val_select_loader = make_loader(X_pcap_vs, X_sar_vs, X_strace_vs, y_val_select, False)
val_reward_loader = make_loader(X_pcap_vr, X_sar_vr, X_strace_vr, y_val_reward, False)
test_loader       = make_loader(X_pcap_te, X_sar_te, X_strace_te, y_te, False)

## Per-seed training: centralized model and weighting schemes

`run_central_seed` trains CAMRA with the RL controller. `run_weighting_comparison_seed` trains the same architecture with fixed class weights, focal loss, or focal loss with the fixed prior.

In [ ]:
def run_central_seed(seed: int, verbose: bool = False,
                     stochastic: Optional[bool] = None,
                     loss_mode: Optional[str] = None,
                     use_prior: Optional[bool] = None) -> Dict:
    """Trains one seed of the central multimodal model with RL-adaptive
    class weighting.

    stochastic : sample from the policy (True) or use its mean (False).
    loss_mode  : "ce" (class-weighted cross-entropy) or "focal" (focal loss
                 with the RL weights as per-class alpha — the RL weights stay
                 in the loop either way, so the policy is never bypassed).
    use_prior  : policy output multiplies the frequency-derived weights
                 (True) or replaces them (False)."""
    loss_mode = CENTRAL_LOSS if loss_mode is None else loss_mode
    use_prior = RL_RESIDUAL_PRIOR if use_prior is None else use_prior

    set_all_seeds(seed)
    model = IoTMalwareDetector(
        pcap_dim=dim_pcap, sar_dim=dim_sar, strace_dim=dim_strace,
        hidden=HIDDEN_DIM, n_classes=N_CLASSES
    ).to(DEVICE)
    optimizer  = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_DL)
    scaler_amp = GradScaler(enabled=USE_AMP)

    prior = bundle["class_weights"].to(DEVICE) if use_prior else None
    rl_agent = RLAgent(n_classes=N_CLASSES, stochastic=stochastic, prior=prior,
                       entropy_beta=RL_ENTROPY_BETA,
                       support=np.bincount(y_val_reward, minlength=N_CLASSES))
    # Epoch 1 has no recall observation yet, so it trains under the prior
    # (or uniform, if no prior) before the policy's first action.
    class_weights = prior.clone() if use_prior else torch.ones(N_CLASSES, device=DEVICE)
    history = defaultdict(list)
    rl_weight_history = []
    best_val_f1, best_epoch = 0.0, 0   # logged only - the FINAL epoch is evaluated,
                                       # matching every other row (no checkpoint selection)

    if verbose:
        print(f"[central] loss={loss_mode}  prior={'freq' if use_prior else 'none'}  "
              f"stochastic={rl_agent.stochastic}")
        print(f"\n{'Epoch':>6} | {'TrainLoss':>10} | {'TrainAcc':>9} | {'ValLoss':>8} | {'ValAcc':>7} | {'ValF1':>7}")
        print("-" * 62)

    for epoch in range(1, EPOCHS_DL + 1):
        if loss_mode == "focal":
            # Default-arg binding freezes THIS epoch's weights into the closure.
            loss_fn = (lambda logits, y, cw=class_weights:
                       focal_loss(logits, y, gamma=FOCAL_LOSS_GAMMA, alpha=cw))
        else:
            loss_fn = None   # train_epoch_multimodal's default: class-weighted CE

        tr_loss, tr_acc = train_epoch_multimodal(model, train_loader, optimizer, scaler_amp,
                                                 class_weights, loss_fn=loss_fn)

        y_pred_vs, y_prob_vs, y_true_vs, val_loss = evaluate_multimodal(model, val_select_loader, return_loss=True)
        val_acc   = float((y_pred_vs == y_true_vs).mean())
        val_f1_pc = f1_score(y_true_vs, y_pred_vs, average=None, zero_division=0, labels=list(range(N_CLASSES)))
        macro_f1  = float(val_f1_pc.mean())

        y_pred_vr, y_prob_vr, y_true_vr, val_reward_loss = evaluate_multimodal(model, val_reward_loader, return_loss=True)
        val_recall_pc = recall_score(y_true_vr, y_pred_vr, average=None, zero_division=0,
                                     labels=list(range(N_CLASSES)))

        class_weights = rl_agent.get_weights(val_recall_pc, epoch / EPOCHS_DL, val_reward_loss).to(DEVICE)
        rl_weight_history.append(class_weights.cpu().numpy())
        rl_agent.store_reward(val_recall_pc)
        rl_agent.update()
        scheduler.step()

        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(val_loss);   history['val_acc'].append(val_acc)
        history['val_f1'].append(macro_f1);     history['lr'].append(optimizer.param_groups[0]['lr'])
        history['val_f1_pc'].append(val_f1_pc.astype(float).tolist())   # per class, val_select

        if macro_f1 > best_val_f1:
            best_val_f1, best_epoch = macro_f1, epoch

        if verbose and (epoch % 5 == 0 or epoch == 1 or epoch == EPOCHS_DL):
            print(f"{epoch:>6} | {tr_loss:>10.4f} | {tr_acc*100:>8.2f}% | {val_loss:>8.4f} | "
                  f"{val_acc*100:>6.2f}% | {macro_f1:>7.4f}")


    y_pred_test, y_prob_test, y_true_test = evaluate_multimodal(model, test_loader)
    metrics = _metrics_from_preds(y_true_test, y_pred_test)
    if verbose:
        print_metrics(y_true_test, y_pred_test, class_names, f"Central Model (seed={seed}, test set)")

    return {
        'seed': seed, 'model': model, 'history': dict(history),
        'y_true': y_true_test, 'y_pred': y_pred_test, 'y_prob': y_prob_test,
        'metrics': metrics, 'best_val_f1': best_val_f1, 'best_epoch': best_epoch,
        'final_epoch': EPOCHS_DL,
        'rl_agent': rl_agent, 'rl_weight_history': rl_weight_history,
    }


def run_weighting_comparison_seed(seed: int, mode: str) -> Dict:
    """Trains the IDENTICAL architecture/epoch budget as run_central_seed,
    swapping only the weighting mechanism: mode='fixed' (static
    bundle["class_weights"]) or mode='focal' (focal loss, no explicit
    weights) - isolates RL's marginal contribution from everything else."""
    set_all_seeds(seed)
    model = IoTMalwareDetector(
        pcap_dim=dim_pcap, sar_dim=dim_sar, strace_dim=dim_strace,
        hidden=HIDDEN_DIM, n_classes=N_CLASSES
    ).to(DEVICE)
    optimizer  = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_DL)
    scaler_amp = GradScaler(enabled=USE_AMP)
    fixed_cw   = bundle["class_weights"].to(DEVICE)
    
    if mode == 'focal':
        loss_fn = lambda logits, y: focal_loss(logits, y, gamma=FOCAL_LOSS_GAMMA)
    elif mode == 'fixed':
        loss_fn = None          # class-weighted cross-entropy with the frequency prior
    elif mode == 'focal_prior':
        # DRL's loss with the RL multiplier removed: focal loss with the frequency
        # prior as a FIXED alpha (m = 1 for every class, every epoch). Isolates the
        # controller's contribution from the prior's.
        loss_fn = lambda logits, y: focal_loss(logits, y, gamma=FOCAL_LOSS_GAMMA, alpha=fixed_cw)
    else:
        raise ValueError(f"unknown weighting mode {mode!r}")
        
    for _ in range(EPOCHS_DL):
        train_epoch_multimodal(model, train_loader, optimizer, scaler_amp, fixed_cw, loss_fn=loss_fn)
        scheduler.step()

    y_pred_test, y_prob_test, y_true_test = evaluate_multimodal(model, test_loader)
    metrics = _metrics_from_preds(y_true_test, y_pred_test)
    del model; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return metrics


def run_modality_holdout_seed(model: nn.Module) -> Dict[str, Dict]:
    """Test-time missing-modality evaluation on an already-trained model -
    zeroes one modality's input at a time (see evaluate_with_modality_holdout
    in Cell 14) and reports the resulting metrics."""
    out = {}
    for mod in MODALITIES:
        yp, ypr, yt = evaluate_with_modality_holdout(model, X_pcap_te, X_sar_te, X_strace_te, y_te, hold_out=mod)
        out[mod] = _metrics_from_preds(yt, yp)
    return out


def run_holdout_generalization_seed(model: nn.Module, verbose: bool = False) -> Optional[Dict]:
    """The held-out arch/malware rows now live INSIDE the real test set
    (idx_te_is_excluded marks which ones), not a separate report - this
    evaluates the full test set once, then splits the result into the
    held-out rows vs. the regular stratified rows, so that leave-one-out
    generalization signal is still visible on its own."""
    if not idx_te_is_excluded.any():
        return None
    yp_all, ypr_all, yt_all = evaluate_multimodal(model, test_loader)
    mask = idx_te_is_excluded
    yt_hold, yp_hold = yt_all[mask], yp_all[mask]
    if verbose:
        print_metrics(yt_hold, yp_hold, class_names,
                       f"Held-out Generalization Test (Arch={EXCLUDE_ARCH!r}, MalwareFamily={EXCLUDE_MALWARE_FAMILY!r})")
    pred_family_counts = pd.Series([class_names[i] for i in yp_hold]).value_counts()
    benign_idx = list(class_names).index("Benign")
    mal = yt_hold != benign_idx
    detection_rate = float((yp_hold[mal] != benign_idx).mean()) if mal.any() else float("nan")
    return {
        'metrics': _metrics_from_preds(yt_hold, yp_hold),
        'predicted_family_counts': pred_family_counts.to_dict(),
        'n_holdout': int(mask.sum()),
        'detection_rate': detection_rate,
    }

## Per-seed ablation and baselines

Single-modality and pairwise models and the four baselines, trained on the same splits and features.

In [ ]:
X_tr_flat = np.hstack([X_pcap_tr, X_sar_tr, X_strace_tr])
X_te_flat = np.hstack([X_pcap_te, X_sar_te, X_strace_te])


def run_ablation_and_baselines_seed(seed: int, verbose: bool = False) -> Dict[str, Dict]:
    """One seed of every non-DRL row in the ablation bar:
    classical + deep baselines, single modalities, modality pairs, and the two
    static weighting schemes. DRL and FL are added by the STEP 6 loop.
    Returns {row_name: metrics_dict} keyed by the ROW_* constants."""
    set_all_seeds(seed)
    results = {}

    # ---- Baselines ----
    results[ROW_RF]  = run_rf_baseline(X_tr_flat, y_tr, X_te_flat, y_te, class_names,
                                       seed=seed, verbose=verbose)
    results[ROW_XGB] = run_xgb_baseline(X_tr_flat, y_tr, X_te_flat, y_te, class_names,
                                        seed=seed, verbose=verbose)
    if RUN_MLP_BASELINE:
        results[ROW_MLP] = run_mlp_baseline(X_tr_flat, y_tr, X_te_flat, y_te, class_names,
                                            bundle["class_weights"], verbose=verbose)
    if RUN_CNN_BASELINE:
        results[ROW_CNN] = run_cnn_baseline(X_tr_flat, y_tr, X_te_flat, y_te, class_names,
                                            bundle["class_weights"], verbose=verbose)

    # ---- Single modalities ----
    modality_data = {"PCAP":   (X_pcap_tr,   X_pcap_te),
                     "SAR":    (X_sar_tr,    X_sar_te),
                     "STRACE": (X_strace_tr, X_strace_te)}
    for mod, (X_mod_tr, X_mod_te) in modality_data.items():
        if X_mod_tr.shape[1] == 0:
            continue
        model = SingleModalModel(X_mod_tr.shape[1], N_CLASSES, hidden=HIDDEN_DIM)
        yp, ypr, yt, _ = train_flat_model(model, X_mod_tr, y_tr, X_mod_te, y_te, EPOCHS_DL,
                                          bundle["class_weights"], tag=ROW_SINGLE[mod], verbose=verbose)
        results[ROW_SINGLE[mod]] = _metrics_from_preds(yt, yp)

    # ---- Modality pairs ----
    for (a, b), row_name in ROW_PAIR.items():
        X_pair_tr = np.hstack([modality_data[a][0], modality_data[b][0]])
        X_pair_te = np.hstack([modality_data[a][1], modality_data[b][1]])
        if X_pair_tr.shape[1] == 0:
            continue
        model = SingleModalModel(X_pair_tr.shape[1], N_CLASSES, hidden=HIDDEN_DIM)
        yp, ypr, yt, _ = train_flat_model(model, X_pair_tr, y_tr, X_pair_te, y_te, EPOCHS_DL,
                                          bundle["class_weights"], tag=row_name, verbose=verbose)
        results[row_name] = _metrics_from_preds(yt, yp)

    # ---- Static weighting schemes (same architecture/epochs as DRL, no RL) ----
    if RUN_FIXED_WEIGHT_COMPARISON:
        results[ROW_FIXED] = run_weighting_comparison_seed(seed, mode='fixed')
    if RUN_FOCAL_LOSS_COMPARISON:
        results[ROW_FOCAL] = run_weighting_comparison_seed(seed, mode='focal')

    return results

## Per-seed federated learning

Four architecture clients initialized from the final centralized model: local training, FedAvg with local BatchNorm statistics, and continued RL class weighting. At test time, each architecture is scored by its own client model.

In [ ]:
def run_fl_seed(seed: int, init_state_dict: dict, rl_agent: RLAgent, verbose: bool = False) -> Dict:
    set_all_seeds(seed)
    global_model = IoTMalwareDetector(
        pcap_dim=dim_pcap, sar_dim=dim_sar, strace_dim=dim_strace,
        hidden=HIDDEN_DIM, n_classes=N_CLASSES
    ).to(DEVICE)
    global_model.load_state_dict(init_state_dict)
    bn_keys = get_batchnorm_keys(global_model) if USE_FEDBN else set()

    arch_of_train_row = bundle["arch_of_row"][idx_tr]
    arch_of_test_row  = bundle["arch_of_row"][idx_te]
    client_names = [a for a in ARCHS if (arch_of_train_row == a).sum() > 0]
    client_row_masks = {a: np.where(arch_of_train_row == a)[0] for a in client_names}
    test_row_masks    = {a: np.where(arch_of_test_row == a)[0] for a in client_names}
    # Rows whose Arch has NO trained client - only non-empty under
    # EXCLUDE_ARCH+"holdout" (a zero-day architecture the federation never
    # trained on). Handled separately below; empty in the normal case.
    zero_day_test_mask = ~np.isin(arch_of_test_row, client_names)

    # FedBN: each client keeps its OWN BatchNorm state, persisted across
    # rounds, initialized from the central model - never synced from the
    # server, never averaged with other clients' (see get_batchnorm_keys).
    client_bn_state = ({a: {k: init_state_dict[k].clone() for k in bn_keys} for a in client_names}
                        if USE_FEDBN else None)

    if verbose:
        print(f"[INFO] FedBN {'ENABLED' if USE_FEDBN else 'disabled'} - BatchNorm stats "
              f"{'stay local per client' if USE_FEDBN else 'are fully aggregated like everything else'}")
        print(f"[INFO] DP noise decay: round 1={fl_noise_scale_for_round(1, FL_ROUNDS):.2e}  "
              f"round {FL_ROUNDS//2}={fl_noise_scale_for_round(FL_ROUNDS//2, FL_ROUNDS):.2e}  "
              f"round {FL_ROUNDS}={fl_noise_scale_for_round(FL_ROUNDS, FL_ROUNDS):.2e}")
        print(f"[INFO] Local LR decay:  round 1={fl_lr_for_round(1, FL_ROUNDS):.2e}  "
              f"round {FL_ROUNDS//2}={fl_lr_for_round(FL_ROUNDS//2, FL_ROUNDS):.2e}  "
              f"round {FL_ROUNDS}={fl_lr_for_round(FL_ROUNDS, FL_ROUNDS):.2e}")
        print("[INFO] FL client sizes (true per-architecture partitions of the TRAIN split):")
        for a in client_names:
            print(f"  {a}: {len(client_row_masks[a])} executions")
        print(f"\n{'Round':>6} | {'GlobalAcc':>10} | {'GlobalF1':>9} | {'GlobalLoss':>11}")
        print("-" * 45)

    fl_history          = {'global_acc': [], 'global_f1': [], 'global_loss': []}
    client_f1_history   = {a: [] for a in client_names}
    client_loss_history = {a: [] for a in client_names}
    client_acc_history  = {a: [] for a in client_names}
    # Start FL from the frequency prior (or uniform when R L_RESIDUAL_PRIOR=False);
    # the agent's own correction takes over from round 1's reward onward.
    fl_class_weights  = rl_agent.prior.detach().cpu().clone()

    client_models, client_sizes = [], []   # populated each round; final round's values kept for post-loop reporting

    for round_ in range(1, FL_ROUNDS + 1):
        client_models, client_sizes = [], []
        round_recall_pc, round_reward_loss = {}, {}
        for arch_name in client_names:
            rows = client_row_masks[arch_name]
            if len(rows) == 0: continue
            local_model = deepcopy(global_model)
            if USE_FEDBN:
                sd = local_model.state_dict()
                for k in bn_keys:
                    sd[k] = client_bn_state[arch_name][k]
                local_model.load_state_dict(sd)
            ds = MultimodalDataset(X_pcap_tr[rows], X_sar_tr[rows], X_strace_tr[rows], y_tr[rows],
                                    augment=USE_AUGMENTATION, noise_std=AUG_NOISE_STD)
            round_noise_scale = fl_noise_scale_for_round(round_, FL_ROUNDS)
            round_lr = fl_lr_for_round(round_, FL_ROUNDS)
            cm = client_local_train(
                local_model,
                DataLoader(ds, BATCH_SIZE, shuffle=True, drop_last=_safe_drop_last(len(ds), True)),
                fl_class_weights, epochs=FL_LOCAL_EPOCHS, lr=round_lr, dp_noise=round_noise_scale
            )
            if USE_FEDBN:
                cm_sd = cm.state_dict()
                client_bn_state[arch_name] = {k: cm_sd[k].clone() for k in bn_keys}
            client_models.append(cm); client_sizes.append(len(rows))

            # Single forward pass per client on the full test set, reused for
            # BOTH client_f1_history (that client's own personalized model,
            # "how would this client's own deployed model do") AND this
            # round's "global" snapshot below - no separate synthetic model.
            cm_dev = cm.to(DEVICE)
            yp_c, _, yt_c, loss_c = evaluate_multimodal(cm_dev, test_loader, return_loss=True)
            client_f1_history[arch_name].append(f1_score(yt_c, yp_c, average='macro', zero_division=0))
            client_loss_history[arch_name].append(loss_c)
            client_acc_history[arch_name].append(float((yp_c == yt_c).mean()))

            # RL reward signal from THIS client's own personalized model on
            # val_reward (not test_loader - same leakage rationale as before).
            yp_r_c, _, yt_r_c, loss_r_c = evaluate_multimodal(cm_dev, val_reward_loader, return_loss=True)
            round_recall_pc[arch_name] = recall_score(yt_r_c, yp_r_c, average=None, zero_division=0,
                                                       labels=list(range(N_CLASSES)))
            round_reward_loss[arch_name] = loss_r_c
            cm.cpu()

        global_model = fedavg_aggregate(global_model, client_models, client_sizes, bn_keys=bn_keys).to(DEVICE)
        global_model.cpu(); gc.collect()

        # "Global" round metrics = size-weighted MEAN of each client's own
        # properly-BN'd personalized model (all just computed above) - NOT a
        # blended-BN forward pass on a model that was never actually trained
        # anywhere. Under FedBN there is no single "global" model to
        # evaluate; the standard convention (Li et al., ICLR 2021) is to
        # report the (weighted) average of the personalized clients. This
        # replaces the previous eval_model/avg_bn snapshot - confirmed to be
        # the source of the ever-climbing, never-plateauing Global Loss:
        # FedBN's per-client BN stats are SUPPOSED to diverge round over
        # round as each client personalizes to its own architecture: a
        # size-weighted average of increasingly-diverged BN stats describes
        # the pooled test set progressively worse, even while every real,
        # individually-evaluated client stayed stable - exactly the "loss
        # climbs steadily while accuracy/F1 stay flat" signature that
        # motivated FedBN in the first place, just relocated into the
        # reporting step instead of the training step.
        total_n = sum(client_sizes)
        g_acc  = sum(client_acc_history[a][-1]  * (n / total_n) for a, n in zip(client_names, client_sizes))
        g_f1   = sum(client_f1_history[a][-1]   * (n / total_n) for a, n in zip(client_names, client_sizes))
        loss_g = sum(client_loss_history[a][-1] * (n / total_n) for a, n in zip(client_names, client_sizes))
        g_recall_pc = sum(round_recall_pc[a] * (n / total_n) for a, n in zip(client_names, client_sizes))
        loss_r      = sum(round_reward_loss[a] * (n / total_n) for a, n in zip(client_names, client_sizes))

        fl_history['global_loss'].append(loss_g)
        fl_history['global_acc'].append(g_acc)
        fl_history['global_f1'].append(g_f1)

        fl_class_weights = rl_agent.get_weights(g_recall_pc, round_ / FL_ROUNDS, loss_r).cpu()
        rl_agent.store_reward(g_recall_pc)
        rl_agent.update()

        if verbose and (round_ % 5 == 0 or round_ == 1 or round_ == FL_ROUNDS):
            print(f"{round_:>6} | {g_acc*100:>9.2f}% | {g_f1:>9.4f} | {loss_g:>11.4f}")

    # ------------------------------------------------------------------
    # Final reported "FL Global Model" predictions: each architecture's OWN
    # test rows scored by ITS OWN final personalized model, concatenated -
    # what a real FedBN deployment looks like (every device runs its own
    # local model; there is no single global model to hand out), not a
    # blended-BN forward pass applied uniformly to the pooled test set.
    # Zero-day rows (an architecture excluded from training entirely, see
    # EXCLUDE_ARCH/"holdout") fall back to the FedAvg body + a size-weighted
    # average of the OTHER clients' BN stats: the best available generic
    # estimate for an architecture the federation never saw, which is
    # exactly the scenario run_holdout_generalization_seed measures.
    global_model = global_model.to(DEVICE)
    yt_parts, yp_parts, ypr_parts = [], [], []

    for arch_name, cm in zip(client_names, client_models):
        rows = test_row_masks[arch_name]
        if len(rows) == 0: continue
        cm_dev = cm.to(DEVICE)
        loader = make_loader(X_pcap_te[rows], X_sar_te[rows], X_strace_te[rows], y_te[rows], shuffle=False)
        yp_a, ypr_a, yt_a = evaluate_multimodal(cm_dev, loader)
        yt_parts.append(yt_a); yp_parts.append(yp_a); ypr_parts.append(ypr_a)
        cm.cpu()

    if zero_day_test_mask.any():
        fallback_model = deepcopy(global_model)
        if USE_FEDBN:
            total_n_final = sum(client_sizes)
            avg_bn = {k: sum(client_bn_state[a][k].float() * (n / total_n_final)
                              for a, n in zip(client_names, client_sizes)).to(client_bn_state[client_names[0]][k].dtype)
                      for k in bn_keys}
            sd = fallback_model.state_dict(); sd.update(avg_bn); fallback_model.load_state_dict(sd)
        rows = np.where(zero_day_test_mask)[0]
        yp_a, ypr_a, yt_a = evaluate_multimodal(fallback_model.to(DEVICE),
            make_loader(X_pcap_te[rows], X_sar_te[rows], X_strace_te[rows], y_te[rows], shuffle=False))
        yt_parts.append(yt_a); yp_parts.append(yp_a); ypr_parts.append(ypr_a)

    yt_fl, yp_fl, ypr_fl = np.concatenate(yt_parts), np.concatenate(yp_parts), np.concatenate(ypr_parts)
    metrics = _metrics_from_preds(yt_fl, yp_fl)
    if verbose:
        print_metrics(yt_fl, yp_fl, class_names, f"FL Global Model (seed={seed}, test set)")

    # Kept for API/backward-compat only (e.g. FLOPs/param-count/latency
    # reporting, which is architecture-only and BN-value-invariant) -
    # loaded with the largest client's final BN purely as a representative
    # snapshot. NOT used for any accuracy/F1/loss number returned above.
    final_model = deepcopy(global_model)
    if USE_FEDBN and client_names:
        largest = client_names[int(np.argmax(client_sizes))]
        sd = final_model.state_dict(); sd.update(client_bn_state[largest]); final_model.load_state_dict(sd)

    return {
        'seed': seed, 'model': final_model, 'fl_history': fl_history,
        'client_f1_history': client_f1_history, 'client_loss_history': client_loss_history,
        'client_acc_history': client_acc_history, 'client_names': client_names,
        'y_true': yt_fl, 'y_pred': yp_fl, 'y_prob': ypr_fl, 'metrics': metrics,
        'global_state': {k: v.detach().cpu() for k, v in global_model.state_dict().items()},
        'client_bn_state': {a: {k: v.detach().cpu() for k, v in sd.items()}
                            for a, sd in client_bn_state.items()} if USE_FEDBN else None,
    }

## Data efficiency and cross-architecture evaluation

Training on 25–100% of the training data, and training on one architecture while testing on each of the others.

In [ ]:
DATA_EFFICIENCY_FRACTIONS = [0.25, 0.5, 0.75, 1.0]

def run_data_efficiency_seed(seed: int, verbose: bool = False) -> Dict[float, Dict]:
    set_all_seeds(seed)
    results = {}
    for frac in DATA_EFFICIENCY_FRACTIONS:
        n_tr = max(1, int(len(y_tr) * frac))
        idx_sub = np.random.choice(len(y_tr), n_tr, replace=False)
        sub_loader = make_loader(X_pcap_tr[idx_sub], X_sar_tr[idx_sub], X_strace_tr[idx_sub], y_tr[idx_sub],
                                  True, USE_AUGMENTATION)
        tmp_model = IoTMalwareDetector(
            pcap_dim=dim_pcap, sar_dim=dim_sar, strace_dim=dim_strace,
            hidden=HIDDEN_DIM, n_classes=N_CLASSES
        ).to(DEVICE)
        opt = torch.optim.AdamW(tmp_model.parameters(), lr=3e-4)
        sc  = GradScaler(enabled=USE_AMP)
        cw  = bundle["class_weights"].to(DEVICE)
        for _ in range(EPOCHS_DL):
            train_epoch_multimodal(tmp_model, sub_loader, opt, sc, cw)
        yp, _, yt = evaluate_multimodal(tmp_model, test_loader)
        results[frac] = _metrics_from_preds(yt, yp)
        if verbose:
            print(f"  {frac*100:>5.0f}% ({n_tr} samples): Acc={results[frac]['accuracy']*100:.2f}%  "
                  f"MacroF1={results[frac]['macro_f1']:.4f}")
        del tmp_model; gc.collect()
    return results


def run_cross_arch_seed(seed: int, verbose: bool = False) -> Tuple[Dict, List[str]]:
    set_all_seeds(seed)
    arch_of_te_row = bundle["arch_of_row"][idx_te]
    archs_eval = [a for a in ARCHS if (bundle["arch_of_row"][idx_tr] == a).sum() >= 20
                  and (arch_of_te_row == a).sum() >= 5]
    if len(archs_eval) < 2:
        return {}, []

    arch_test_f1 = {ta: {te: 0.0 for te in archs_eval} for ta in archs_eval}
    for train_arch in archs_eval:
        rows_tr = np.where(bundle["arch_of_row"][idx_tr] == train_arch)[0]
        local_model = IoTMalwareDetector(
            pcap_dim=dim_pcap, sar_dim=dim_sar, strace_dim=dim_strace,
            hidden=HIDDEN_DIM, n_classes=N_CLASSES
        ).to(DEVICE)
        opt = torch.optim.AdamW(local_model.parameters(), lr=3e-4)
        sc  = GradScaler(enabled=USE_AMP)
        cw  = bundle["class_weights"].to(DEVICE)
        ds_tr = MultimodalDataset(X_pcap_tr[rows_tr], X_sar_tr[rows_tr], X_strace_tr[rows_tr], y_tr[rows_tr],
                                   augment=USE_AUGMENTATION, noise_std=AUG_NOISE_STD)
        ld_tr = DataLoader(ds_tr, BATCH_SIZE, shuffle=True, drop_last=_safe_drop_last(len(ds_tr), True))
        for _ in range(EPOCHS_DL):
            train_epoch_multimodal(local_model, ld_tr, opt, sc, cw)

        for test_arch in archs_eval:
            rows_te = np.where(arch_of_te_row == test_arch)[0]
            ld_te = DataLoader(
                MultimodalDataset(X_pcap_te[rows_te], X_sar_te[rows_te], X_strace_te[rows_te], y_te[rows_te]),
                BATCH_SIZE, shuffle=False
            )
            ypred, _, ytrue = evaluate_multimodal(local_model, ld_te)
            arch_test_f1[train_arch][test_arch] = f1_score(ytrue, ypred, average='macro', zero_division=0)
            if verbose:
                print(f"  {train_arch:>8} -> {test_arch:>8}: MacroF1={arch_test_f1[train_arch][test_arch]:.4f}")
        del local_model; gc.collect()
    return arch_test_f1, archs_eval

## Pre-flight checks

Confirms the configuration before the multi-seed run starts.

In [ ]:
print(f"[PRE-FLIGHT] run tag = {_run_tag()!r}  (Arch={EXCLUDE_ARCH!r}, Family={EXCLUDE_MALWARE_FAMILY!r})")
assert CENTRAL_LOSS == "focal" and RL_ALPHA_MIN == 10.0 and RL_RESIDUAL_PRIOR
assert RUN_FIXED_WEIGHT_COMPARISON and RUN_FOCAL_LOSS_COMPARISON
assert 'RUN_FOCAL_FIXED_COMPARISON' not in globals() and 'RUN_DRL_NO_FOCAL' not in globals(), \
    "stale flags still defined — restart the kernel after editing Cell 4"
print(f"{len(SEEDS)} seeds, {len(EXPECTED_ROWS)} ablation rows:", EXPECTED_ROWS)

## Optional: Focal Loss + Prior control

Trains focal loss with the fixed frequency prior (CAMRA's loss without the RL controller) and compares it with the DRL, Focal Loss, and Fixed Weighting results of a completed base run. Off by default; set `RUN_FOCAL_PRIOR_CONTROL = True` and point `PREV_RESULTS_JSON` at the base run's results.

In [ ]:
# ═══════════════════════════════════════════════════════════
# OPTIONAL CONTROL — Focal Loss + Prior (no RL). Off by default.
# ═══════════════════════════════════════════════════════════
RUN_FOCAL_PRIOR_CONTROL = False         # base configuration only
ROW_FOCAL_PRIOR         = "Focal Loss + Prior"
CONTROL_SEEDS           = SEEDS         # smoke test first with SEEDS[:1]

if RUN_FOCAL_PRIOR_CONTROL:
    from scipy.stats import wilcoxon

    # --- previous results must come from a completed base run ----------------
    prev = json.load(open(PREV_RESULTS_JSON))
    prev_pd = json.load(open(PREV_RESULTS_JSON.parent.parent / "artifacts" / "plot_data.json"))
    assert "val_f1_pc" in prev_pd["central_histories"][0], \
        "PREV_RESULTS_JSON must point at the results of a completed base run"
    assert EXCLUDE_ARCH is None and EXCLUDE_MALWARE_FAMILY is None, "base configuration only"
    # Check the DATA, not just the flags: a kernel whose data prep ran under a holdout
    # config keeps that split even after the flags are edited back to None.
    assert not np.any(idx_te_is_excluded), \
        "test split contains held-out rows - data prep ran under a holdout config; restart the kernel"
    _oth = list(class_names).index("Other") if "Other" in list(class_names) else None
    if _oth is not None:
        assert (y_te == _oth).sum() < MIN_TEST_SUPPORT, "test split is not the base split"
    assert prev["exclusion_config"]["exclude_arch"] is None \
        and prev["exclusion_config"]["exclude_malware_family"] is None
    assert prev["seeds"] == list(SEEDS)

    # --- train the control ----------------------------------------------------
    ctrl, ctrl_times = [], []
    for seed in CONTROL_SEEDS:
        t0 = time.time()
        m = run_weighting_comparison_seed(seed, mode='focal_prior')
        ctrl_times.append(time.time() - t0)
        ctrl.append({k: m[k] for k in ("macro_f1", "macro_f1_all", "weighted_f1",
                                       "accuracy", "per_class_f1", "per_class_support")})
        assert m["per_class_support"].get("Other", 0) < MIN_TEST_SUPPORT, \
            f"evaluated set is not the base test set: support={m['per_class_support']}"
        print(f"[seed {seed}] {ROW_FOCAL_PRIOR}: macro-F1={m['macro_f1']:.4f}  "
              f"all-class={m['macro_f1_all']:.4f}  ({ctrl_times[-1]:.0f}s)")

    # --- attribution contrasts (Holm within this family of three) -------------
    raw = prev["raw_per_seed"]
    idx = [list(SEEDS).index(s) for s in CONTROL_SEEDS]
    fp  = [c["macro_f1"] for c in ctrl]
    pick = lambda row: [raw[row]["macro_f1"][i] for i in idx]

    def _boot(a, b, n_boot=N_BOOTSTRAP, ci=BOOTSTRAP_CI, rng_seed=12345):   # identical to STEP 7
        d = np.asarray(a, float) - np.asarray(b, float)
        bm = d[np.random.default_rng(rng_seed).integers(0, len(d), (n_boot, len(d)))].mean(1)
        lo, hi = np.quantile(bm, [(1 - ci) / 2, 1 - (1 - ci) / 2])
        return float(d.mean()), float(lo), float(hi), d

    def _holm(p):
        o, adj, run = np.argsort(p), np.empty(len(p)), 0.0
        for r, i in enumerate(o):
            run = max(run, (len(p) - r) * p[i]); adj[i] = min(1.0, run)
        return adj.tolist()

    family = [
        ("controller (RL multiplier)", ROW_DRL,         pick(ROW_DRL),   ROW_FOCAL_PRIOR, fp),
        ("frequency prior",            ROW_FOCAL_PRIOR, fp,              ROW_FOCAL,       pick(ROW_FOCAL)),
        ("focal vs CE, same prior",    ROW_FOCAL_PRIOR, fp,              ROW_FIXED,       pick(ROW_FIXED)),
    ]
    rows, pv = [], []
    for label, an, a, bn, b in family:
        md, lo, hi, d = _boot(a, b)
        try:
            _, p = wilcoxon(a, b, zero_method='wilcox', alternative='two-sided')
        except ValueError:
            p = 1.0
        rows.append({"isolates": label, "a": an, "b": bn, "mean_a": float(np.mean(a)),
                     "mean_b": float(np.mean(b)), "mean_diff": md, "ci_low": lo, "ci_high": hi,
                     "p_raw": float(p), "n_seeds_a_better": int((d > 0).sum())})
        pv.append(float(p))

    print(f"\n{ROW_FOCAL_PRIOR}: {np.mean(fp):.4f} ± {np.std(fp, ddof=0) if len(fp) > 1 else float('nan'):.4f}")
    for r, ph in zip(rows, _holm(pv)):
        r["p_holm"] = ph
        sig = "significant" if (ph < 0.05 and not r["ci_low"] <= 0 <= r["ci_high"]) else "n.s."
        print(f"  {r['isolates']:<28} {r['a'][:27]:<27} - {r['b'][:20]:<20} Δ={r['mean_diff']:+.4f} "
              f"[{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]  p_holm={ph:.4f}  {sig}  "
              f"({r['n_seeds_a_better']}/{len(idx)})")
    # Per-seed identity: (DRL - Focal) = (DRL - F+P) + (F+P - Focal), so the two effects add up exactly.

    with open(OUT_DIR / "reports" / "focal_prior_control.json", "w") as f:
        json.dump({"row": ROW_FOCAL_PRIOR, "seeds": list(CONTROL_SEEDS),
                   "definition": "focal loss (gamma=FOCAL_LOSS_GAMMA) with the frequency prior as a fixed "
                                 "per-class alpha; DRL's loss with the RL multiplier fixed at 1",
                   "per_seed": ctrl, "training_time_seconds": ctrl_times,
                   "contrasts": rows,
                   "multiplicity_correction": "Holm-Bonferroni within these three contrasts; "
                                              "separate from the six-contrast primary family"},
                  f, indent=2, default=lambda o: o.tolist() if hasattr(o, "tolist") else float(o))
    print(f"[INFO] saved -> {OUT_DIR / 'reports' / 'focal_prior_control.json'}")

## Multi-seed execution

Runs every configuration for each seed. The models of the representative seed are kept for the complexity report and the diagnostic plots; all metrics are collected for every seed.

In [ ]:
all_central   = []
all_fl        = []
ablation_by_config      = defaultdict(list)
dataeff_by_frac         = defaultdict(list)
crossarch_by_pair       = defaultdict(list)
crossarch_archs_seen    = set()
modality_holdout_by_mod = defaultdict(list)
holdout_gen_results     = []
training_times          = defaultdict(list)
representative          = {}   # populated only when seed == REPRESENTATIVE_SEED

for seed in tqdm(SEEDS, desc="Seeds"):
    is_repr = (seed == REPRESENTATIVE_SEED)
    if is_repr:
        print(f"\n{'='*70}\n  Representative seed {seed} - verbose output shown for this one only\n{'='*70}")

    # ---- Central (DRL) ----
    central_out, t_central = time_call(run_central_seed, seed, verbose=is_repr)
    all_central.append({'seed': seed, 'metrics': central_out['metrics'],
                         'history': central_out['history'], 'best_val_f1': central_out['best_val_f1']})
    ablation_by_config[ROW_DRL].append(central_out['metrics'])
    training_times['DRL'].append(t_central)
    central_state = deepcopy(central_out['model'].state_dict())
    print(f"[seed {seed}] DRL evaluated at epoch {central_out['final_epoch']} "
          f"(val_select all-class peak was epoch {central_out['best_epoch']}, not used)")
    if SAVE_PER_SEED_STATE:
        np.savez_compressed(ARTIFACT_DIR / f"preds_drl_seed{seed}{_run_tag()}.npz",
                            y_true=central_out['y_true'], y_pred=central_out['y_pred'],
                            y_prob=central_out['y_prob'])
        torch.save({k: v.detach().cpu() for k, v in central_state.items()},
                   ARTIFACT_DIR / f"state_drl_seed{seed}{_run_tag()}.pt")

    if RUN_MODALITY_HOLDOUT_EVAL:
        for mod, metrics in run_modality_holdout_seed(central_out['model']).items():
            modality_holdout_by_mod[mod].append(metrics)
    modality_holdout_by_mod['Full (no holdout)'].append(central_out['metrics'])

    hg = run_holdout_generalization_seed(central_out['model'], verbose=is_repr)
    if hg is not None:
        holdout_gen_results.append(hg)

    if is_repr:
        representative.update({
            'central_model': central_out['model'],
            'central_y_true': central_out['y_true'], 'central_y_pred': central_out['y_pred'],
            'central_y_prob': central_out['y_prob'],
            'rl_agent': central_out['rl_agent'], 'rl_weight_history': central_out['rl_weight_history'],
        })
    else:
        del central_out['model']

    # ---- Federated (continues the SAME seed's RL policy across rounds) ----
    fl_out, t_fl = time_call(run_fl_seed, seed, central_state, central_out['rl_agent'], verbose=is_repr)
    all_fl.append({'seed': seed, 'metrics': fl_out['metrics'], 'fl_history': fl_out['fl_history']})
    ablation_by_config[ROW_FL].append(fl_out['metrics'])
    training_times['FL'].append(t_fl)
    if SAVE_PER_SEED_STATE:
        np.savez_compressed(ARTIFACT_DIR / f"preds_fl_seed{seed}{_run_tag()}.npz",
                            y_true=fl_out['y_true'], y_pred=fl_out['y_pred'], y_prob=fl_out['y_prob'])
        torch.save({'global': fl_out['global_state'], 'client_bn': fl_out['client_bn_state']},
                   ARTIFACT_DIR / f"state_fl_seed{seed}{_run_tag()}.pt")

    if is_repr:
        representative.update({
            'fl_model': fl_out['model'], 'fl_y_true': fl_out['y_true'], 'fl_y_pred': fl_out['y_pred'],
            'fl_y_prob': fl_out['y_prob'], 'client_f1_history': fl_out['client_f1_history'],
            'client_loss_history': fl_out['client_loss_history'],
            'client_acc_history': fl_out['client_acc_history'],
            'client_names': fl_out['client_names'],
        })
    else:
        del fl_out['model']

    if RERUN_DRL_FL_ONLY:
        ablation_out = {}        # spliced in from PREV_RESULTS_JSON after the loop
    else:
        # ---- Ablation + baselines (classical + deep) + weighting-scheme comparison ----
        ablation_out, t_ablation = time_call(run_ablation_and_baselines_seed, seed, verbose=is_repr)
        clash = set(ablation_out) & {ROW_DRL, ROW_FL}
        assert not clash, f"ablation returned rows that STEP 6 already owns: {clash}"
        for config_name, metrics in ablation_out.items():
            ablation_by_config[config_name].append(metrics)
        training_times['Ablation+baselines (all configs)'].append(t_ablation)

        # ---- Data efficiency ----
        dataeff_out, t_dataeff = time_call(run_data_efficiency_seed, seed, verbose=False)
        for frac, metrics in dataeff_out.items():
            dataeff_by_frac[frac].append(metrics)
        training_times['Data efficiency (4 fractions)'].append(t_dataeff)

        # ---- Cross-architecture ----
        (arch_test_f1, archs_eval), t_crossarch = time_call(run_cross_arch_seed, seed, verbose=False)
        crossarch_archs_seen.update(archs_eval)
        for ta in archs_eval:
            for te in archs_eval:
                crossarch_by_pair[(ta, te)].append(arch_test_f1[ta][te])
        training_times['Cross-architecture (all pairs)'].append(t_crossarch)

    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        
    print(f"[seed {seed}] " + "  ".join(
        f"{n}={m['macro_f1']:.4f}" for n, m in
        [(ROW_DRL, central_out['metrics']), (ROW_FL, fl_out['metrics'])] + sorted(ablation_out.items())))

print(f"\n[INFO] Completed {len(SEEDS)} seed(s): {SEEDS}")

## Optional: merge with a previous run

Used only when `RERUN_DRL_FL_ONLY = True`: the DRL-dependent configurations are rerun and all other results are taken from a previous run of the same configuration. Off by default.

In [ ]:
# ═══════════════════════════════════════════════════════════
# MERGE — splice unchanged rows from the previous run of this configuration
# ═══════════════════════════════════════════════════════════
PREV_DATAEFF = PREV_CROSSARCH = PREV_TRAIN_TIMES = None
if RERUN_DRL_FL_ONLY:
    prev = json.load(open(PREV_RESULTS_JSON))
    # Guard against splicing numbers from a different configuration.
    pcfg = prev["exclusion_config"]
    assert prev["seeds"] == list(SEEDS), f"seed mismatch: {prev['seeds']} vs {SEEDS}"
    assert prev["raw_per_seed"]["_seed_order"] == list(SEEDS)
    assert pcfg["exclude_arch"] == EXCLUDE_ARCH and pcfg["exclude_malware_family"] == EXCLUDE_MALWARE_FAMILY, \
        f"PREV_RESULTS_JSON is from a different run: {pcfg}"

    n = len(SEEDS)
    spliced = []
    for row, rec in prev["raw_per_seed"].items():
        if row.startswith("_") or row in (ROW_DRL, ROW_FL):
            continue
        ablation_by_config[row] = [
            {"macro_f1": rec["macro_f1"][i], "macro_f1_all": rec["macro_f1_all"][i],
             "weighted_f1": rec["weighted_f1"][i], "accuracy": rec["accuracy"][i],
             "per_class_f1": rec["per_class_f1"][i]}
            for i in range(n)]
        spliced.append(row)
    PREV_DATAEFF     = prev.get("data_efficiency", {})
    PREV_CROSSARCH   = prev.get("cross_architecture", {})
    PREV_TRAIN_TIMES = json.load(open(PREV_COMPLEXITY_JSON))["training_time_seconds"]
    print(f"[MERGE] rerun: {[ROW_DRL, ROW_FL]}  |  spliced from previous run: {sorted(spliced)}")

## Aggregate results

Mean ± standard deviation across seeds for every configuration, holdout breakdowns, and the full results report.

In [ ]:
def mean_std(values: List[float]) -> Tuple[float, float]:
    arr = np.array(values, dtype=np.float64)
    return float(arr.mean()), float(arr.std(ddof=0))

def agg_metric_list(metric_dicts: List[Dict], key: str = 'macro_f1') -> Tuple[float, float]:
    return mean_std([d.get(key, float('nan')) for d in metric_dicts])
    
n_seeds = len(SEEDS)
_got = set(ablation_by_config)
_missing, _extra = set(EXPECTED_ROWS) - _got, _got - set(EXPECTED_ROWS)
if _missing: print(f"[WARN] rows missing from the ablation: {sorted(_missing)}")
if _extra:   print(f"[WARN] unexpected ablation rows (would appear as extra bars): {sorted(_extra)}")
assert all(len(v) == n_seeds for v in ablation_by_config.values()), \
    "some ablation rows do not have one entry per seed"

print(f"\n{'='*70}\n  AGGREGATED RESULTS (mean ± std over {n_seeds} seed{'s' if n_seeds>1 else ''})\n{'='*70}")

raw_per_seed = {
    name: {**{metric: [float(d.get(metric, float('nan'))) for d in dicts]
              for metric in ('macro_f1', 'macro_f1_all', 'weighted_f1', 'accuracy')},
           'per_class_f1': [d.get('per_class_f1', {}) for d in dicts]}
    for name, dicts in ablation_by_config.items()
}
raw_per_seed["_seed_order"] = list(SEEDS)
raw_per_seed["_primary_metric"] = f"macro-F1 over classes with >= {MIN_TEST_SUPPORT} test samples"


print("\n-- Central vs. Federated vs. Baselines vs. Ablation (Macro-F1) --")
ablation_summary = {}   # name -> (mean, std), for plot_ablation_radar
all_model_summary = {}  # name -> {'macro_f1':, 'macro_f1_std':, 'weighted_f1':, 'weighted_f1_std':, 'accuracy':}
for name, metric_dicts in sorted(ablation_by_config.items(), key=lambda kv: -agg_metric_list(kv[1])[0]):
    mf1_m, mf1_s = agg_metric_list(metric_dicts, 'macro_f1')
    wf1_m, wf1_s = agg_metric_list(metric_dicts, 'weighted_f1')
    acc_m, acc_s = agg_metric_list(metric_dicts, 'accuracy')
    all_m, all_s = agg_metric_list(metric_dicts, 'macro_f1_all')
    ablation_summary[name] = (mf1_m, mf1_s)
    all_model_summary[name] = {'macro_f1': mf1_m, 'macro_f1_std': mf1_s,
                                'weighted_f1': wf1_m, 'weighted_f1_std': wf1_s,
                                'accuracy': acc_m, 'accuracy_std': acc_s,
                                'macro_f1_all': all_m, 'macro_f1_all_std': all_s}
    print(f"  {name:<32} Macro-F1: {mf1_m:.4f} ± {mf1_s:.4f}   Acc: {acc_m*100:.2f}% ± {acc_s*100:.2f}%  Macro-F1 (all classes): {all_m:.4f} ± {all_s:.4f}")

print("\n-- DRL vs. static weighting schemes (same architecture and epoch budget) --")
for name in [ROW_DRL, ROW_FIXED, ROW_FOCAL]:
    if name in ablation_summary:
        m, s = ablation_summary[name]
        print(f"  {name:<32} Macro-F1: {m:.4f} ± {s:.4f}")

print("\n-- Modality holdout at test time (Macro-F1) --")
modality_holdout_summary = {}
full_f1_summary = agg_metric_list(modality_holdout_by_mod['Full (no holdout)'], 'macro_f1')
print(f"  {'Full (no holdout)':<20} Macro-F1: {full_f1_summary[0]:.4f} ± {full_f1_summary[1]:.4f}")
for mod in MODALITIES:
    if mod in modality_holdout_by_mod:
        m, s = agg_metric_list(modality_holdout_by_mod[mod], 'macro_f1')
        modality_holdout_summary[mod] = (m, s)
        drop = full_f1_summary[0] - m
        print(f"  Missing {mod:<12} Macro-F1: {m:.4f} ± {s:.4f}   (drop: {drop:+.4f})")

print("\n-- Data efficiency (Macro-F1 by training-data fraction) --")
dataeff_summary = {}
for frac in DATA_EFFICIENCY_FRACTIONS:
    if frac in dataeff_by_frac:
        mf1_m, mf1_s = agg_metric_list(dataeff_by_frac[frac], 'macro_f1')
        acc_m, acc_s = agg_metric_list(dataeff_by_frac[frac], 'accuracy')
        dataeff_summary[frac] = {'macro_f1': (mf1_m, mf1_s), 'accuracy': (acc_m, acc_s)}
        print(f"  {frac*100:>5.0f}%   Macro-F1: {mf1_m:.4f} ± {mf1_s:.4f}   Acc: {acc_m*100:.2f}% ± {acc_s*100:.2f}%")

print("\n-- Cross-architecture generalization (Macro-F1, diagonal = same-arch train/test) --")
archs_eval_sorted = sorted(crossarch_archs_seen)
crossarch_summary = {}
for ta in archs_eval_sorted:
    for te in archs_eval_sorted:
        if (ta, te) in crossarch_by_pair:
            crossarch_summary[(ta, te)] = mean_std(crossarch_by_pair[(ta, te)])
if archs_eval_sorted:
    for a in archs_eval_sorted:
        if (a, a) in crossarch_summary:
            m, s = crossarch_summary[(a, a)]
            print(f"  {a:<8} (same-arch)   Macro-F1: {m:.4f} ± {s:.4f}")

if holdout_gen_results:
    print("\n-- Held-out generalization test (EXCLUDE_ARCH / EXCLUDE_MALWARE_FAMILY) --")
    hg_mf1 = mean_std([h['metrics']['macro_f1'] for h in holdout_gen_results])
    hg_acc = mean_std([h['metrics']['accuracy'] for h in holdout_gen_results])
    print(f"  Macro-F1: {hg_mf1[0]:.4f} ± {hg_mf1[1]:.4f}   Acc: {hg_acc[0]*100:.2f}% ± {hg_acc[1]*100:.2f}%")
    print(f"  ({holdout_gen_results[0]['n_holdout']} held-out execution(s), "
          f"Arch={EXCLUDE_ARCH!r} MalwareFamily={EXCLUDE_MALWARE_FAMILY!r})")

if RERUN_DRL_FL_ONLY:
    # Data efficiency / cross-arch were not rerun: restore their summaries.
    dataeff_summary = {float(k): {'macro_f1': (v['macro_f1_mean'], v['macro_f1_std']),
                                  'accuracy': (float('nan'), float('nan'))}
                       for k, v in PREV_DATAEFF.items()}
    crossarch_summary = {tuple(k.split('->')): (v['macro_f1_mean'], v['macro_f1_std'])
                         for k, v in PREV_CROSSARCH.items()}
    archs_eval_sorted = sorted({a for pair in crossarch_summary for a in pair})

# Persist the full aggregated report
with open(OUT_DIR / "reports" / "multiseed_results.json", "w") as f:
    json.dump({
        "seeds": SEEDS,
        "raw_per_seed": raw_per_seed,
        "exclusion_config": {"exclude_arch": EXCLUDE_ARCH, "exclude_malware_family": EXCLUDE_MALWARE_FAMILY,
                              "exclusion_mode": EXCLUSION_MODE},
        "ablation_and_baselines": {k: {"macro_f1_mean": v[0], "macro_f1_std": v[1]}
                                    for k, v in ablation_summary.items()},
        "modality_holdout": {k: {"macro_f1_mean": v[0], "macro_f1_std": v[1]}
                              for k, v in modality_holdout_summary.items()},
        "data_efficiency": {str(k): {"macro_f1_mean": v['macro_f1'][0], "macro_f1_std": v['macro_f1'][1]}
                             for k, v in dataeff_summary.items()},
        "cross_architecture": {f"{ta}->{te}": {"macro_f1_mean": v[0], "macro_f1_std": v[1]}
                                for (ta, te), v in crossarch_summary.items()},
        # Previously computed and printed but never saved here - this is the
        # direct evidence that the leave-one-out holdout mechanism (arch
        # and/or malware family) actually ran and produced real numbers on
        # rows that were genuinely absent from training.
        "holdout_generalization": ({
            "n_holdout": holdout_gen_results[0]['n_holdout'],
            "macro_f1_mean": mean_std([h['metrics']['macro_f1'] for h in holdout_gen_results])[0],
            "macro_f1_std": mean_std([h['metrics']['macro_f1'] for h in holdout_gen_results])[1],
            "accuracy_mean": mean_std([h['metrics']['accuracy'] for h in holdout_gen_results])[0],
            "accuracy_std": mean_std([h['metrics']['accuracy'] for h in holdout_gen_results])[1],
            "detection_rate_mean": mean_std([h['detection_rate'] for h in holdout_gen_results])[0],
            "detection_rate_std":  mean_std([h['detection_rate'] for h in holdout_gen_results])[1],
            "predicted_class_counts_total": pd.DataFrame([h['predicted_family_counts'] for h in holdout_gen_results])
                                              .fillna(0).sum().astype(int).to_dict(),
        } if holdout_gen_results else None),
        "training_time_seconds": {**(PREV_TRAIN_TIMES or {}),
                                  **{k: {"mean": float(np.mean(v)), "std": float(np.std(v, ddof=0))}
                                     for k, v in training_times.items()}},
    }, f, indent=2)
print(f"\n[INFO] Full aggregated report saved -> {OUT_DIR / 'reports' / 'multiseed_results.json'}")

## Save plot data

Writes all figure data to disk before plotting, so the figures can be regenerated without retraining.

In [ ]:
# ═══════════════════════════════════════════════════════════
# SAVE PLOT DATA — runs before any figure, so a plotting error
# cannot lose the artifacts from the multi-seed run
# ═══════════════════════════════════════════════════════════
save_eval_artifacts("central", representative['central_y_true'],
                    representative['central_y_pred'], representative['central_y_prob'], class_names)
save_eval_artifacts("fl_global", representative['fl_y_true'],
                    representative['fl_y_pred'], representative['fl_y_prob'], class_names)

np.savez_compressed(ARTIFACT_DIR / f"rl_dynamics{_run_tag()}.npz",
                    reward_history=np.asarray(representative['rl_agent'].reward_history),
                    recall_history=np.asarray(representative['rl_agent'].recall_history),
                    weight_history=np.asarray(representative['rl_weight_history']),
                    class_names=np.asarray(class_names, dtype=object),
                    n_central_steps=len(representative['rl_weight_history']))

_agent = representative['rl_agent']
np.savez_compressed(ARTIFACT_DIR / f"policy_diagnostics{_run_tag()}.npz",
                    alpha0=np.asarray(_agent.alpha0_history),
                    entropy=np.asarray(_agent.entropy_history),
                    explore_gap=np.asarray(_agent.explore_gap_history),
                    alpha=np.asarray(_agent.alpha_history),
                    n_central_steps=len(representative['rl_weight_history']))

plot_data = {
    "run_tag": _run_tag(),
    "class_names": list(class_names),
    "seeds": list(SEEDS),
    "representative_seed": REPRESENTATIVE_SEED,
    # complete per-seed histories: training curves, FL global curves, central vs FL
    "central_histories": [c['history'] for c in all_central],
    "fl_histories":      [f['fl_history'] for f in all_fl],
    # FL client heatmap (representative seed)
    "client_f1_history":   representative.get('client_f1_history'),
    "client_loss_history": representative.get('client_loss_history'),
    "client_acc_history":  representative.get('client_acc_history'),
    # ablation bar, macro vs weighted, modality radar
    "ablation_summary":  {k: list(v) for k, v in ablation_summary.items()},
    "all_model_summary": all_model_summary,
    # modality holdout bar, including the "Complete" bar
    "modality_holdout_summary": {k: list(v) for k, v in modality_holdout_summary.items()},
    "full_f1_summary": list(full_f1_summary),
}
with open(ARTIFACT_DIR / f"plot_data{_run_tag()}.json", "w") as f:
    json.dump(plot_data, f, default=float)
print(f"[ARTIFACT] eval arrays, RL dynamics, policy diagnostics, plot_data{_run_tag()}.json -> {ARTIFACT_DIR}")

## Reproducibility manifests

Saves the split manifest (hash, architecture, family, and split of every execution) and the pre-processing manifest (selected features, label mapping, class weights).

In [ ]:
# ═══════════════════════════════════════════════════════════
# REPRODUCIBILITY MANIFESTS
# ═══════════════════════════════════════════════════════════
split_order = [("train", idx_tr), ("val_select", idx_val_select),
               ("val_reward", idx_val_reward), ("test", idx_te)]
rows = []
for split_name, ix in split_order:
    rows.append(pd.DataFrame({
        "hash":   bundle["hash_of_row"][ix],
        "arch":   bundle["arch_of_row"][ix],
        "family": full_df["MalwareFamily"].to_numpy()[ix],
        "label":  bundle["y"][split_name],
        "split":  split_name,
    }))
manifest = pd.concat(rows, ignore_index=True)
# The test block is last in `split_order`, so idx_te_is_excluded lines up with
# the final len(idx_te) rows of the concatenated frame.
manifest["is_holdout_row"] = False
manifest.iloc[len(manifest) - len(idx_te):,
              manifest.columns.get_loc("is_holdout_row")] = idx_te_is_excluded
manifest.to_csv(MANIFEST_DIR / f"split_manifest{_run_tag()}.csv", index=False)

with open(MANIFEST_DIR / f"preprocessing_manifest{_run_tag()}.json", "w") as f:
    json.dump({
        "retained_hashes_n": int(manifest["hash"].nunique()),
        "selected_features": {mod: list(cols) for mod, cols in bundle["selected_cols"].items()},
        "selected_feature_counts": {mod: len(cols) for mod, cols in bundle["selected_cols"].items()},
        "total_selected_features": int(sum(len(c) for c in bundle["selected_cols"].values())),
        "requested_feature_budget": TOTAL_FEATURE_BUDGET,
        "modality_feature_share": MODALITY_FEATURE_SHARE,
        "log1p_transformed_columns": list(bundle["log_cols"]),
        "label_encoding": {name: int(i) for i, name in enumerate(bundle["class_names"])},
        "class_weight_power": CLASS_WEIGHT_POWER,
        "class_weights": {n: float(w) for n, w in zip(bundle["class_names"],
                                                      bundle["class_weights"].tolist())},
        "split_seed": SEED,
        "split_sizes": {n: int(len(ix)) for n, ix in split_order},
        "stratification": "hybrid (Arch, MalwareFamily) joint where cell >= "
                          f"{SPARSE_CELL_JOINT_THRESHOLD}, else MalwareFamily only",
        "exclusion": {"arch": EXCLUDE_ARCH, "malware_family": EXCLUDE_MALWARE_FAMILY,
                      "mode": EXCLUSION_MODE},
        "seeds": SEEDS,
    }, f, indent=2)

print(f"[MANIFEST] split_manifest{_run_tag()}.csv  ({len(manifest)} rows, "
      f"{manifest['hash'].nunique()} unique hashes)")
print(f"[MANIFEST] preprocessing_manifest{_run_tag()}.json  "
      f"({sum(len(c) for c in bundle['selected_cols'].values())} features selected "
      f"against a budget of {TOTAL_FEATURE_BUDGET})")

## Paired statistics

Seed-paired comparisons of the main configurations: bootstrap confidence intervals, Wilcoxon signed-rank tests, and Holm correction.

In [ ]:
# ═══════════════════════════════════════════════════════════
# RAW PER-SEED VALUES + PAIRED STATISTICAL COMPARISON
# ═══════════════════════════════════════════════════════════
from scipy.stats import wilcoxon

def paired_bootstrap_ci(a, b, n_boot=N_BOOTSTRAP, ci=BOOTSTRAP_CI, rng_seed=12345):
    """Bootstrap CI on the paired per-seed difference a - b. Pairing is valid
    because the split is fixed across seeds (see `set_all_seeds`): every config
    is scored on the identical test set, and seed i differs only in training
    randomness, so seed i of a and seed i of b are matched observations."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    rng = np.random.default_rng(rng_seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    boot_means = d[idx].mean(axis=1)
    lo, hi = np.quantile(boot_means, [(1 - ci) / 2, 1 - (1 - ci) / 2])
    return float(d.mean()), float(lo), float(hi), d


def holm_bonferroni(pvals: List[float]) -> List[float]:
    """Holm step-down adjustment — the contrasts below are all drawn from one
    family of comparisons on one test set, so uncorrected p-values would
    overstate significance."""
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m, dtype=float)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * pvals[i])
        adj[i] = min(1.0, running)
    return adj.tolist()


print(f"\n{'='*94}\n  PAIRED COMPARISONS — macro-F1, {len(SEEDS)} paired seeds, fixed split\n{'='*94}")
rows, pvals = [], []
for a_name, b_name in STAT_CONTRASTS:
    if a_name not in raw_per_seed or b_name not in raw_per_seed:
        print(f"[SKIP] {a_name} vs {b_name}: not present in this run")
        continue
    a = raw_per_seed[a_name]['macro_f1']
    b = raw_per_seed[b_name]['macro_f1']
    mean_d, lo, hi, d = paired_bootstrap_ci(a, b)
    try:
        stat, p = wilcoxon(a, b, zero_method='wilcox', alternative='two-sided')
    except ValueError:      # all differences identical / zero
        stat, p = float('nan'), 1.0
    rows.append({"a": a_name, "b": b_name,
                 "mean_a": float(np.mean(a)), "mean_b": float(np.mean(b)),
                 "mean_diff": mean_d, "ci_low": lo, "ci_high": hi,
                 "wilcoxon_stat": float(stat), "p_raw": float(p),
                 "n_seeds_a_better": int((d > 0).sum())})
    pvals.append(float(p))

for row, p_adj in zip(rows, holm_bonferroni(pvals)):
    row["p_holm"] = p_adj
    crosses_zero = row["ci_low"] <= 0.0 <= row["ci_high"]
    verdict = "n.s." if (crosses_zero or p_adj >= 0.05) else "significant"
    print(f"  {row['a'][:30]:<30} - {row['b'][:30]:<30} "
          f"Δ={row['mean_diff']:+.4f}  95% CI [{row['ci_low']:+.4f}, {row['ci_high']:+.4f}]  "
          f"p_holm={p_adj:.4f}  {verdict}  ({row['n_seeds_a_better']}/{len(SEEDS)} seeds favour first)")
print(f"{'='*94}")

with open(OUT_DIR / "reports" / f"paired_statistics{_run_tag()}.json", "w") as f:
    json.dump({"seeds": SEEDS, "n_bootstrap": N_BOOTSTRAP, "ci": BOOTSTRAP_CI,
               "pairing_basis": "fixed split across seeds; only training randomness varies",
               "multiplicity_correction": "Holm-Bonferroni within this contrast family",
               "contrasts": rows, "raw_per_seed": raw_per_seed}, f, indent=2)
print(f"[INFO] Paired statistics saved -> reports/paired_statistics{_run_tag()}.json")

## Computational complexity and communication cost

Model size, FLOPs, and inference latency (measured on the representative seed), training time per stage, and the federated communication cost.

In [ ]:
# ═══════════════════════════════════════════════════════════
# COMPLEXITY (INCL. RF/XGB) + FULL-DUPLEX COMMUNICATION COST
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*70}\n  COMPUTATIONAL COMPLEXITY\n{'='*70}")

sample_multimodal = (torch.randn(1, dim_pcap, device=DEVICE),
                     torch.randn(1, dim_sar, device=DEVICE),
                     torch.randn(1, dim_strace, device=DEVICE))
sample_flat = (torch.randn(1, X_tr_flat.shape[1], device=DEVICE),)

complexity_report = {}
complexity_report['CAMRA'] = report_model_complexity(
    representative['central_model'], sample_multimodal, "CAMRA")
if RUN_MLP_BASELINE:
    complexity_report['MLP'] = report_model_complexity(
        MLPBaseline(X_tr_flat.shape[1], N_CLASSES, hidden=HIDDEN_DIM * 2).to(DEVICE),
        sample_flat, "MLP")
if RUN_CNN_BASELINE:
    complexity_report['CNN'] = report_model_complexity(
        TabularCNN(X_tr_flat.shape[1], N_CLASSES).to(DEVICE), sample_flat, "CNN")

# --- Random Forest and XGBoost, profiled on CPU with wall-clock timing.
# Refit here on the representative seed rather than threading the fitted object
# out of the ablation loop; RF/XGB fit in seconds on this feature set.
rf_profile_model = RandomForestClassifier(n_estimators=100, n_jobs=-1,
                                          random_state=REPRESENTATIVE_SEED).fit(X_tr_flat, y_tr)
complexity_report['Random Forest'] = report_sklearn_complexity(
    rf_profile_model, X_te_flat, "Random Forest")

if HAS_XGB:
    y_tr_local_prof, _ = _remap_labels_for_xgb(y_tr)
    xgb_profile_model = xgb.XGBClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=4, eval_metric='mlogloss',
        random_state=REPRESENTATIVE_SEED, tree_method='hist').fit(X_tr_flat, y_tr_local_prof)
    complexity_report['XGBoost'] = report_sklearn_complexity(
        xgb_profile_model, X_te_flat, "XGBoost")

print(f"\n-- Training time per config (mean ± std over {n_seeds} seeds) --")
for name, times in training_times.items():
    m, s = float(np.mean(times)), float(np.std(times, ddof=0))
    print(f"  {name:<38} {m:>8.1f}s ± {s:>6.1f}s")

# ── Communication cost ──────────────────────────────────────────
# The previous table counted client->server uploads only. Every round is
# full duplex: the server also broadcasts the aggregated model back to every
# client. Under FedBN the BatchNorm entries are never transmitted in either
# direction, so the payload is smaller than the full serialized model; both
# numbers are reported so the difference is explicit.
bn_keys_repr = get_batchnorm_keys(representative['central_model']) if USE_FEDBN else set()
full_model_mb = get_model_size_mb(representative['central_model'])
payload_mb    = state_payload_mb(representative['central_model'], exclude_keys=bn_keys_repr)
n_clients     = len(representative['client_names'])

upload_per_round   = payload_mb * n_clients
download_per_round = payload_mb * n_clients
round_trip         = upload_per_round + download_per_round
total_transfer     = round_trip * FL_ROUNDS

print(f"\n-- FL communication volume ({FL_ROUNDS} rounds, {n_clients} clients) --")
print(f"  Full serialized model                 : {full_model_mb:.3f} MB")
print(f"  Transmitted payload per transfer      : {payload_mb:.3f} MB "
      f"({'BatchNorm excluded under FedBN' if USE_FEDBN else 'all parameters'})")
print(f"  Client -> server, per round           : {upload_per_round:.3f} MB")
print(f"  Server -> client, per round           : {download_per_round:.3f} MB")
print(f"  Round trip, per round                 : {round_trip:.3f} MB")
print(f"  Total over {FL_ROUNDS} rounds                  : {total_transfer:.1f} MB")
print(f"  Per client, total                     : {2 * payload_mb * FL_ROUNDS:.1f} MB")
print(f"  NOTE: lower bound. Excludes TLS/protocol framing, retransmission under")
print(f"        intermittent connectivity, secure-aggregation masking overhead,")
print(f"        and any straggler re-broadcast.")

with open(OUT_DIR / "reports" / f"computational_complexity{_run_tag()}.json", "w") as f:
    json.dump({
        "models": complexity_report,
        "training_time_seconds": {**(PREV_TRAIN_TIMES or {}),   # sections not rerun
                                  **{k: {"mean": float(np.mean(v)), "std": float(np.std(v, ddof=0))}
                                     for k, v in training_times.items()}},
        "communication": {
            "full_model_mb": full_model_mb,
            "transmitted_payload_mb": payload_mb,
            "fedbn_excludes_batchnorm": bool(USE_FEDBN),
            "n_clients": n_clients,
            "rounds": FL_ROUNDS,
            "upload_per_round_mb": upload_per_round,
            "download_per_round_mb": download_per_round,
            "round_trip_per_round_mb": round_trip,
            "total_transfer_mb": total_transfer,
            "per_client_total_mb": 2 * payload_mb * FL_ROUNDS,
            "excluded_from_estimate": ["protocol/TLS framing", "retransmission",
                                       "secure aggregation", "straggler re-broadcast"],
        },
    }, f, indent=2)
print(f"\n[INFO] Complexity report saved -> reports/computational_complexity{_run_tag()}.json")

## Plots

Diagnostic plots for the representative seed (UMAP, confusion matrices, ROC/PR, RL dynamics) and mean ± standard deviation plots across seeds (training and federated curves, ablation and modality-holdout charts).

In [ ]:
X_pcap_all    = np.vstack([X_pcap_tr, X_pcap_vs, X_pcap_vr, X_pcap_te])
X_sar_all     = np.vstack([X_sar_tr, X_sar_vs, X_sar_vr, X_sar_te])
X_strace_all  = np.vstack([X_strace_tr, X_strace_vs, X_strace_vr, X_strace_te])
y_all_splits  = np.concatenate([y_tr, y_val_select, y_val_reward, y_te])

arch_of_train_row = bundle["arch_of_row"][idx_tr]
client_row_masks_repr = {a: np.where(arch_of_train_row == a)[0] for a in representative['client_names']}

#plot_global_distribution(y_all_splits, class_names)
# plot_client_distributions({a: y_tr[client_row_masks_repr[a]] for a in representative['client_names']}, class_names)
umap_scores = plot_umap_raw_and_embedding(
    {"PCAP": X_pcap_all, "SAR": X_sar_all, "STRACE": X_strace_all},
    representative['central_model'], X_pcap_all, X_sar_all, X_strace_all, y_all_splits, class_names
)

plot_training_curves([c['history'] for c in all_central])
plot_confusion_matrix(representative['central_y_true'], representative['central_y_pred'], class_names, "Central")
plot_roc_curves(representative['central_y_true'], representative['central_y_prob'], class_names, 'central_roc')
plot_pr_curves(representative['central_y_true'], representative['central_y_prob'], class_names, 'central_pr')
plot_per_class_radar(representative['central_y_true'], representative['central_y_pred'], class_names, "Central")
# plot_rl_dynamics(representative['rl_agent'].reward_history, representative['rl_weight_history'],
                  # class_names, "Central")
from types import SimpleNamespace
n_central = len(representative['rl_weight_history'])   # central epochs; later steps are FL rounds
_agent = representative['rl_agent']
plot_rl_dynamics(_agent.reward_history[:n_central], representative['rl_weight_history'],
                 class_names, "Central")
plot_rl_dynamics(_agent.reward_history[n_central:], [], class_names, "FL")   # reward only
plot_policy_diagnostics(SimpleNamespace(alpha0_history=_agent.alpha0_history[:n_central],
                                        entropy_history=_agent.entropy_history[:n_central],
                                        explore_gap_history=_agent.explore_gap_history[:n_central]),
                        "Central")

plot_fl_global([f['fl_history'] for f in all_fl])

# ── FL client "drift" plots (representative seed) ───────────────────────
# The raw per-round signal is genuinely noisy - FedBN + only
# FL_LOCAL_EPOCHS local steps/round + DP noise all inject real round-to-
# round variance - so a bare line plot is dominated by that noise. A
# rolling mean makes the underlying trend readable without hiding the
# variance, which is kept visible as a light line behind it.
ROLL_WIN = max(3, FL_ROUNDS // 10)

def _rolling(vals, win):
    return pd.Series(vals).rolling(win, min_periods=1, center=True).mean().to_numpy()

for arch_name, vals in representative['client_f1_history'].items():
    vals = np.array(vals)
    rounds = np.arange(1, len(vals) + 1)
    plt.figure(figsize=(5, 4))
    plt.plot(rounds, vals, color='lightgray', linewidth=1, marker='o',
              markersize=max(2, SCATTER_POINT_SIZE - 2), alpha=0.6, label='Per-round')
    plt.plot(rounds, _rolling(vals, ROLL_WIN), color='tab:blue', linewidth=1,
              label=f'{ROLL_WIN}-round rolling mean')
    plt.title(f'FL Drift - {arch_name}')
    plt.xlabel('Round'); plt.ylabel('Macro-F1'); plt.legend(fontsize=8)
    plt.grid(axis='x', linestyle='--', alpha=0.5, linewidth=0.5)
    _savefig(f"fl_drift_{arch_name.lower()}")

# Compact multi-architecture summary: Architecture x Round heatmap (already
# defined above, previously never called) reads much more legibly than 4
# separate noisy line charts when comparing clients against each other.
plot_fl_client_heatmap(representative['client_f1_history'])

# All 4 clients' rolling trends on one axis, for direct side-by-side
# comparison of drift behavior across architectures.
plt.figure(figsize=(5, 4))
for arch_name, vals in representative['client_f1_history'].items():
    plt.plot(np.arange(1, len(vals) + 1), _rolling(np.array(vals), ROLL_WIN), linewidth=1, label=arch_name)
plt.title(f'FL Client Drift - Rolling Mean')
plt.xlabel('Round'); plt.ylabel('Macro-F1 (rolling mean)'); plt.legend()
_savefig("fl_drift_all_archs_rolling")

# Diagnostic (uses only already-recorded data, no retraining): size-
# weighted mean of clients' own personalized F1 vs. this run's recorded
# "Global F1"/"Global Loss" (computed via the old blended-BN eval_model -
# see run_fl_seed, now fixed for future runs). A visible gap here is
# exactly what the ever-climbing Global Loss plot was flagging: the
# blended snapshot describing the pooled test set progressively worse than
# any real personalized client, even while every individual client stayed
# stable.
fl_history_by_seed = {f['seed']: f['fl_history'] for f in all_fl}
client_sizes_repr  = np.array([len(client_row_masks_repr[a]) for a in representative['client_names']])
w = client_sizes_repr / client_sizes_repr.sum()
n_rounds_repr = len(next(iter(representative['client_f1_history'].values())))
mean_client_f1 = np.zeros(n_rounds_repr)
for a, wi in zip(representative['client_names'], w):
    mean_client_f1 += wi * np.array(representative['client_f1_history'][a])
recorded_global_f1 = np.array(fl_history_by_seed[REPRESENTATIVE_SEED]['global_f1'][:n_rounds_repr])

plt.figure(figsize=(8, 4.5))
plt.plot(np.arange(1, n_rounds_repr + 1), mean_client_f1, linewidth=2,
          label="Size-weighted mean of clients' own F1")
plt.plot(np.arange(1, n_rounds_repr + 1), recorded_global_f1, linewidth=2, linestyle='--',
          label='Recorded "Global F1" (blended-BN eval, this run)')
plt.title('Personalized Clients vs. Blended-BN "Global" Snapshot', fontweight='bold')
plt.xlabel('Round'); plt.ylabel('Macro-F1'); plt.legend(fontsize=8)
_savefig("fl_client_vs_blended_global_f1")

plot_confusion_matrix(representative['fl_y_true'], representative['fl_y_pred'], class_names, "FL_Global")
plot_roc_curves(representative['fl_y_true'], representative['fl_y_prob'], class_names, 'fl_global_roc')
plot_pr_curves(representative['fl_y_true'], representative['fl_y_prob'], class_names, 'fl_global_pr')
plot_per_class_radar(representative['fl_y_true'], representative['fl_y_pred'], class_names, "FL_Global")

plot_ablation_radar(ablation_summary, all_model_summary)
plot_central_vs_fl([c['history'] for c in all_central], [f['fl_history'] for f in all_fl])

if modality_holdout_summary:
    plot_modality_holdout(modality_holdout_summary, full_f1_summary)

## Summary

In [ ]:
print("\n" + "="*70)
print(f"  FINAL RESULTS SUMMARY (mean ± std over {n_seeds} seed{'s' if n_seeds>1 else ''})")
print("="*70)
c_m, c_s = ablation_summary[ROW_DRL]
f_m, f_s = ablation_summary[ROW_FL]
print(f"  {'DRL (central)':<26} ->  Macro-F1: {c_m:.4f} ± {c_s:.4f}")
print(f"  {'FL (federated)':<26} ->  Macro-F1: {f_m:.4f} ± {f_s:.4f}")
for name in [ROW_FIXED, ROW_FOCAL, ROW_RF, ROW_XGB, ROW_MLP, ROW_CNN]:
    if name in ablation_summary:
        m, s = ablation_summary[name]
        print(f"  {name:<26} ->  Macro-F1: {m:.4f} ± {s:.4f}")
print("="*70)
print(f"\n[PIPELINE COMPLETE]  All outputs in '{OUT_DIR}/'")
print(f"Plots saved to       '{OUT_DIR}/plots/'")
print(f"Reports saved to     '{OUT_DIR}/reports/'  (alignment coverage, skew, UMAP scores,")
print(f"                       multi-seed results, computational complexity)")

## Package outputs (Kaggle)

Zips the output folder for download.

In [ ]:
import os
import shutil
from IPython.display import FileLink

# Define paths
output_dir = "/kaggle/working/results"  # Change to your output folder name or '/kaggle/working/outputs_Final'
zip_name = f"outputs{_run_tag() or '_base'}.zip"

# Create a zip archive
if os.path.exists(output_dir):
    shutil.make_archive(base_name=zip_name[:-4], format="zip", root_dir=output_dir)
    print(f"Compressed {output_dir} into {zip_name} ({os.path.getsize(zip_name) / (1024 * 1024):.2f} MB)")
    display(FileLink(zip_name))
else:
    print(f"Directory '{output_dir}' not found.")